<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# 🔍 Build a RAG Pipeline: Your AI Research Assistant

You have validated the platform (notebook 00) and loaded models (notebook 01).
Now you build the thing those exist for: a retrieval system that reads a corpus
of research papers and answers questions from them, with sources.

The corpus here is the 2025–2026 literature on **reasoning post-training** —
teaching a small model to reason, either by reinforcement learning on verifiable
rewards or by distilling a larger model's reasoning. Notebook 03 then has agents
debate that question using this index.

## 🎯 What You'll Build

- **Fetch** papers from arXiv — and reach the foundational ones that search alone cannot find
- **Extract** six kinds of evidence from each paper, not just its abstract
- **Chunk and embed** them with your own embedding model, through the LLM Gateway
- **Store** the vectors in Qdrant with payloads you can filter on
- **Retrieve** a deliberate mix of explanation and evidence
- **Trace** every query in Langfuse, and use those traces to find real defects

## 💡 Why RAG

A language model knows what it was trained on, states it with equal confidence
whether or not it is right, and cannot tell you where it came from. RAG changes
the question from *what do you know* to *what do these passages say* — so the
answer is grounded in documents you chose, stays current as you add papers, and
arrives with citations you can check.

## 🏗️ The Pipeline

```mermaid
graph TB
    A[📄 arXiv: metadata, sources, bibliographies] --> A2[⚓ Anchors<br/>named + discovered by citation]
    A2 --> B[✂️ Chunk by content type<br/>evidence types capped]
    B --> C[🧮 Embed in batches<br/>via LLM Gateway]
    C --> D[💾 Qdrant<br/>vectors + filterable payload]

    E[❓ Question] --> F[🧮 Embed query]
    F --> G[🔍 Balanced retrieval<br/>4 prose + 2 evidence]
    G --> H[🤖 Generate from context]
    H --> J[✨ Answer + sources]

    D -.->|filtered similarity search| G
    K[📊 Langfuse] -.->|traces| H

    style A fill:#e1f5ff,color:#1a1a1a
    style A2 fill:#e1f5ff,color:#1a1a1a
    style E fill:#e1f5ff,color:#1a1a1a
    style D fill:#e8f5e9,color:#1a1a1a
    style H fill:#fff4e6,color:#1a1a1a
    style J fill:#c8e6c9,color:#1a1a1a
    style K fill:#fce4ec,color:#1a1a1a
```

## 🔬 What makes this more than a toy

Most RAG tutorials index abstracts, retrieve the top five chunks, and stop. Each
of those choices quietly limits what the system can answer, and this notebook
shows the limit and the fix:

| the usual shortcut | what it costs | what this does instead |
|---|---|---|
| index abstracts only | claims without the conditions they were measured under | six content types, including experimental setup and limitations |
| trust keyword search | the paper that founded the field never appears | anchors named by hand and discovered from bibliographies |
| index everything found | 70% of the index is number tables | evidence types capped per paper |
| retrieve plain top-k | a conceptual question answered by a benchmark grid | reserved slots for prose and evidence |
| trust the defaults | a query vector unrelated to the query | the tiktoken and thinking-budget defects, explained in section 6 |

Sections are ordered so each fix follows the symptom that motivates it.

---

**Prerequisites**
- ✅ `00-platform-validation.ipynb` completed
- ✅ `01-load-models.ipynb` completed — a chat model **and** an embedding model loaded
- ⏱️ First run takes roughly 10 minutes, most of it arXiv's rate limit; later runs use the cache

---
## Setup

**What**: Initialize helper functions, connect to the LLM Gateway via `tk-llm`, and load platform service credentials from environment variables.

**Why**: We need to connect to these Thinkube platform services:
- **LLM Gateway** - Routes to your local chat model and embedding model via `tk-llm`
- **Qdrant** - Vector database for fast similarity search
- **Langfuse** - Observability and tracing for debugging

**How**: The `tk-llm` SDK auto-discovers the gateway. Other credentials come from environment variables injected by JupyterHub.

In [11]:
import os
from IPython.display import display, HTML
from tk_llm import LLMClient, get_openai_client

# Helper functions for colored output
def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

# Initialize tk-llm management client and OpenAI-compatible inference client
# Using distinct names to avoid conflicts with Langfuse's get_client()
llm_mgmt = LLMClient()
oai_client = get_openai_client()

# Discover loaded models
available = llm_mgmt.list_models(state="available")
CHAT_MODEL = next((m.id for m in available.models if m.task == "text-generation"), None)
EMBED_MODEL = next((m.id for m in available.models if m.task == "feature-extraction"), None)

# Platform services
QDRANT_URL = os.environ.get('QDRANT_URL')
LANGFUSE_HOST = os.environ.get('LANGFUSE_HOST')
LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY')

info(f"Chat model: {CHAT_MODEL or 'NONE — load one in notebook 01'}")
info(f"Embedding model: {EMBED_MODEL or 'NONE — load one in notebook 01'}")
info(f"Qdrant URL: {QDRANT_URL}")
info(f"Langfuse host: {LANGFUSE_HOST}")

---
## Where this data comes from, and what stays on your machine

This notebook downloads other people's work onto the machine it runs on. Two
things follow from that, and neither depends on where you are.

**Metadata and abstracts** come from the arXiv API. arXiv dedicates its metadata
to the public domain, and the notebook follows arXiv's Terms of Use: one request
per three seconds, and the attribution line the fetch cell prints. Bulk harvesting
belongs on arXiv's S3 and OAI-PMH channels, not here.

**LaTeX sources** are different, and that is why `FETCH_FULL_TEXT` defaults to off.
Each paper carries its own licence — CC-BY, CC-BY-NC-SA, CC0, or arXiv's default
non-exclusive licence — and the arXiv API does not report which. Whether copying
one is permitted rests on a research or text-and-data-mining exception, and those
differ by country: the EU has one, the United States reasons through fair use,
and some jurisdictions have neither. Turning the flag on is a decision to make
knowingly.

**Everything fetched stays local.** `latex_cache/` and the `*_cache_rl.pkl` files
hold copyrighted text; the vectors in Qdrant are derived from it. Analysing them
privately is one thing, republishing them is another. Do not commit these files,
ship them in an image, or share a database snapshot — the repository's
`.gitignore` already excludes them, and it should stay that way.

If you add a citation source later, its terms travel with it: OpenAlex releases
its data under CC0 and asks nothing, while Semantic Scholar's graph is ODC-BY and
requires attribution wherever you use it.

---
## 1. Fetch Papers from ArXiv

**What**: Collect the papers the rest of the notebook reasons over.

**Why**: A retrieval system is only as good as what it indexes, and *how* you
collect decides what it can later answer. Three searches run here, one per side
of the debate plus background, so neither side enters the argument better armed
than the other. A corpus assembled from a single query would let the pipeline
confirm whatever that query implied.

**How**:

1. **Anchors by identifier.** A handful of papers are named outright, because
   search cannot find them — the next section explains why, and how the corpus
   discovers the rest for itself.
2. **Three keyword searches**, 50 results each, deduplicated against everything
   already collected.
3. **Optionally, the sources.** With `FETCH_FULL_TEXT` on, each paper's LaTeX is
   downloaded once and cached, and five kinds of content are extracted from it.

**What gets extracted, and why each earns its place**

| content type | what it holds | what it answers |
|---|---|---|
| `abstract` | the paper's own summary | what is claimed |
| `algorithm` | pseudocode blocks | how the method works |
| `results` | tables of numbers | how well it performed |
| `equation` | the formal statement | what it optimises |
| `setup` | experimental-setup prose | *on which model sizes, budgets and rewards* |
| `discussion` | limitations and conclusions | *where it breaks* |

The last two matter more than they look. An abstract claims, a table measures —
but only the setup section says the numbers came from a 70B model when your
question is about a 7B one, and only the limitations section admits where the
method fails. A debate about which method to use is decided on exactly that kind
of evidence, so it is indexed as its own type and can be searched for directly.

The bibliographies (`.bbl`, `.bib`) are kept too. They arrive in the same
archive and cost no extra request; section 2 shows what they are for.

**Rate limiting**: one request per three seconds, as arXiv's Terms of Use ask,
and skipped entirely on a cache hit. Fetching 150 papers takes about eight
minutes the first time and seconds thereafter.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Load cached data (skip if you want fresh ArXiv fetch)
# ═══════════════════════════════════════════════════════════════════════════════
# Run this cell to load previously cached papers, chunks, and embeddings.
# This skips the slow ArXiv fetch and LaTeX download (~10+ minutes).
# If cache files don't exist, this cell does nothing - proceed to next cell.

import pickle
import os

PAPERS_CACHE = "papers_cache_rl.pkl"
CHUNKS_CACHE = "chunks_cache_rl.pkl"
EMBEDDINGS_CACHE = "embeddings_cache_rl.pkl"


def load_cache(path):
    """Return the cached object, or None when the file is missing or unreadable.

    An interrupted run can leave an empty or truncated cache file behind; that
    file is removed here so the next cells rebuild what it should have held.
    """
    if not os.path.exists(path):
        return None
    try:
        if os.path.getsize(path) == 0:
            raise EOFError("empty cache file")
        with open(path, 'rb') as f:
            return pickle.load(f)
    except (EOFError, pickle.UnpicklingError, AttributeError) as e:
        info(f"  Cache file {path} is unreadable ({e}) — removed; it will be rebuilt")
        os.remove(path)
        return None


# Check if cache exists
papers = load_cache(PAPERS_CACHE)
if papers is not None:
    success(f"Loaded {len(papers)} papers from cache")
    
    # Count by category
    grpo_count = len([p for p in papers if p.get('category') == 'grpo'])
    distill_count = len([p for p in papers if p.get('category') == 'distill'])
    general_count = len([p for p in papers if p.get('category') == 'general'])
    papers_with_latex = len([p for p in papers if p.get('latex_source')])
    
    info(f"  - GRPO/RLVR papers: {grpo_count}")
    info(f"  - Distillation papers: {distill_count}")
    info(f"  - General reasoning papers: {general_count}")
    info(f"  - Papers with LaTeX source: {papers_with_latex}")
    
    # Load chunks if available
    chunks = load_cache(CHUNKS_CACHE)
    if chunks is not None:
        success(f"Loaded {len(chunks)} chunks from cache")
        info("  → Skip cell 2 (chunking) and go to cell 3 (embeddings) or cell 4 (Qdrant)")
    
    # Load embeddings if available
    embeddings = load_cache(EMBEDDINGS_CACHE)
    if embeddings is not None:
        success(f"Loaded {len(embeddings)} embeddings from cache")
        info("  → Skip to cell 4 (Store in Qdrant)")
    
    info("\n💡 TIP: If you have all 3 caches, skip directly to cell 4 (Store in Qdrant)")
else:
    info("No cache found - run the next cell to fetch papers from ArXiv")
    info("(This will take ~10+ minutes due to ArXiv rate limiting)")

In [ ]:
import arxiv
import time
import tarfile
import io
import re
import requests
import os

# ═══════════════════════════════════════════════════════════════════════════════
# Fetch papers from ArXiv with BALANCED queries for the GRPO vs Distillation debate
# ═══════════════════════════════════════════════════════════════════════════════
info("Fetching papers from ArXiv with balanced coverage for debate...")

# Query 1: GRPO / RL with verifiable rewards (Pro-GRPO evidence)
query_grpo = '"group relative policy optimization" OR "GRPO" OR "reinforcement learning with verifiable rewards" OR "RLVR" OR "reinforcement learning from verifiable rewards"'

# Query 2: Reasoning distillation from a teacher (Pro-Distillation evidence)
query_distill = '"reasoning distillation" OR "chain-of-thought distillation" OR "distilling reasoning" OR ("knowledge distillation" AND "reasoning" AND "language model")'

# Query 3: General reasoning post-training papers (background knowledge)
query_general = '("large language model" AND "reasoning") AND ("reinforcement learning" OR "distillation" OR "post-training")'

client = arxiv.Client()
papers = []

# ═══════════════════════════════════════════════════════════════════════════════
# Full-text extraction is OPT-IN
# ═══════════════════════════════════════════════════════════════════════════════
# Abstracts come from arXiv's metadata, which arXiv dedicates to the public
# domain. LaTeX sources do not: each paper carries its own licence (CC-BY,
# CC-BY-NC-SA, CC0, or arXiv's default non-exclusive licence), and the arXiv
# API does not report which. Downloading a source is a copy, and what makes
# that copy lawful — a research or text-and-data-mining exception — differs by
# country and does not exist everywhere.
#
# So this is a deliberate choice, not a default. With it off you get abstracts:
# enough for retrieval and for the debate's opening arguments. With it on you
# also get algorithms, results tables, equations, and the setup and limitations
# prose that let the debaters weigh evidence instead of claims.
#
# Either way the files stay on this machine. Do not redistribute latex_cache/
# or the pickles: they hold other people's copyrighted work.
FETCH_FULL_TEXT = False

# ═══════════════════════════════════════════════════════════════════════════════
# LaTeX Cache Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# Cache raw LaTeX files locally to avoid re-downloading from ArXiv during debugging.
# This significantly speeds up iteration when modifying the LaTeX processing logic.

LATEX_CACHE_DIR = "latex_cache"
os.makedirs(LATEX_CACHE_DIR, exist_ok=True)
info(f"LaTeX cache directory: {LATEX_CACHE_DIR}/")

def get_latex_cache_path(paper_id):
    """Get the cache file path for a paper's LaTeX source."""
    # Clean paper_id (remove version suffix like v1, v2)
    clean_id = paper_id.split('v')[0] if 'v' in paper_id else paper_id
    return os.path.join(LATEX_CACHE_DIR, f"{clean_id}.tex")

# A cached source written before bibliographies were kept is missing the half
# of the file the citation analysis needs, so the marker doubles as a version
# stamp: no marker, no cache hit, and the paper is fetched again.
BIB_SEPARATOR = "\n%%THINKUBE-BIBLIOGRAPHY%%\n"


def load_latex_from_cache(paper_id):
    """Load LaTeX source from local cache, if it was written by this version."""
    cache_path = get_latex_cache_path(paper_id)
    if os.path.exists(cache_path):
        with open(cache_path, 'r', encoding='utf-8', errors='ignore') as f:
            cached = f.read()
        if BIB_SEPARATOR in cached:
            return cached
    return None


def split_source(tex_content):
    """Return (body, bibliography) for a cached source."""
    if not tex_content:
        return "", ""
    body, _, bib = tex_content.partition(BIB_SEPARATOR)
    return body, bib

def save_latex_to_cache(paper_id, tex_content):
    """Save LaTeX source to local cache for future runs."""
    if tex_content:
        cache_path = get_latex_cache_path(paper_id)
        with open(cache_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(tex_content)

# ═══════════════════════════════════════════════════════════════════════════════
# Helper functions for LaTeX source extraction
# ═══════════════════════════════════════════════════════════════════════════════

def download_latex_source(paper_id):
    """Download and extract LaTeX source from ArXiv (with local caching)."""
    # Check cache first
    cached = load_latex_from_cache(paper_id)
    if cached:
        return cached
    
    # Clean paper_id (remove version suffix like v1, v2)
    clean_id = paper_id.split('v')[0] if 'v' in paper_id else paper_id
    url = f"https://arxiv.org/e-print/{clean_id}"
    
    try:
        response = requests.get(url, timeout=30, headers={'User-Agent': 'Thinkube-Research-Assistant/1.0'})
        if response.status_code != 200:
            return None
        
        # Try to extract .tex files from tar.gz
        tex_content = None
        try:
            tar = tarfile.open(fileobj=io.BytesIO(response.content), mode='r:gz')
            tex_files, bib_files = [], []
            for member in tar.getmembers():
                f = tar.extractfile(member)
                if not f:
                    continue
                # The .bbl and .bib files hold the reference list. They travel in
                # the same archive, so keeping them costs no extra request, and
                # they are what makes a local citation graph possible.
                if member.name.endswith('.tex'):
                    tex_files.append(f.read().decode('utf-8', errors='ignore'))
                elif member.name.endswith(('.bbl', '.bib')):
                    bib_files.append(f.read().decode('utf-8', errors='ignore'))
            tex_content = ('\n'.join(tex_files) + BIB_SEPARATOR + '\n'.join(bib_files)) if tex_files else None
        except tarfile.TarError:
            # Single file (not tar.gz) - try direct decode
            try:
                tex_content = response.content.decode('utf-8', errors='ignore') + BIB_SEPARATOR
            except:
                tex_content = None
        
        # Save to cache for future runs
        if tex_content:
            save_latex_to_cache(paper_id, tex_content)
        
        return tex_content
    except Exception as e:
        return None

def latex_to_plain(text):
    """Convert LaTeX to readable plain text for better embeddings.
    
    LaTeX content has poor semantic similarity with natural language queries
    because embeddings don't understand LaTeX syntax. Converting to plain text
    dramatically improves retrieval quality for results tables and algorithms.
    """
    if not text:
        return text
    
    # Remove LaTeX formatting commands
    text = re.sub(r'\\textbf\{([^}]*)\}', r'\1', text)
    text = re.sub(r'\\textit\{([^}]*)\}', r'\1', text)
    text = re.sub(r'\\emph\{([^}]*)\}', r'\1', text)
    text = re.sub(r'\\underline\{([^}]*)\}', r'\1', text)
    text = re.sub(r'\\makecell\{([^}]*)\}', r'\1', text)
    text = re.sub(r'\\multirow\{[^}]*\}\{[^}]*\}\{([^}]*)\}', r'\1', text)
    text = re.sub(r'\\multicolumn\{[^}]*\}\{[^}]*\}\{([^}]*)\}', r'\1', text)
    
    # Remove citations and references
    text = re.sub(r'\\cite\{[^}]*\}', '', text)
    text = re.sub(r'\\ref\{[^}]*\}', '', text)
    text = re.sub(r'\\label\{[^}]*\}', '', text)
    
    # Remove table structure commands
    text = re.sub(r'\\toprule|\\midrule|\\bottomrule|\\hline|\\cline\{[^}]*\}', '', text)
    text = re.sub(r'\\begin\{tabular\}\{[^}]*\}', '', text)
    text = re.sub(r'\\end\{tabular\}', '', text)
    
    # Convert table delimiters to readable format
    text = text.replace('&', ' | ')
    text = text.replace('\\\\', '\n')
    
    # Remove other common LaTeX commands
    text = re.sub(r'\\[a-zA-Z]+\*?\{([^}]*)\}', r'\1', text)  # Generic \command{text} → text
    text = re.sub(r'\\[a-zA-Z]+\*?', '', text)  # Remove standalone commands
    
    # Clean up whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\n\s*\n', '\n\n', text)  # Normalize paragraph breaks
    
    return text

def extract_algorithms(tex_content):
    """Extract algorithm environments from LaTeX."""
    if not tex_content:
        return []
    
    patterns = [
        r'\\begin\{algorithm\}(.*?)\\end\{algorithm\}',
        r'\\begin\{algorithm\*\}(.*?)\\end\{algorithm\*\}',
        r'\\begin\{algorithmic\}(.*?)\\end\{algorithmic\}',
        r'\\begin\{lstlisting\}(.*?)\\end\{lstlisting\}',
        r'\\begin\{minted\}.*?\}(.*?)\\end\{minted\}',
    ]
    algorithms = []
    for pattern in patterns:
        matches = re.findall(pattern, tex_content, re.DOTALL | re.IGNORECASE)
        algorithms.extend(matches)
    return algorithms

def extract_tables(tex_content):
    """Extract tables with results (likely contain performance metrics)."""
    if not tex_content:
        return []
    
    patterns = [
        r'\\begin\{table\}(.*?)\\end\{table\}',
        r'\\begin\{table\*\}(.*?)\\end\{table\*\}',
        r'\\begin\{tabular\}(.*?)\\end\{tabular\}',
    ]
    tables = []
    for pattern in patterns:
        matches = re.findall(pattern, tex_content, re.DOTALL | re.IGNORECASE)
        tables.extend(matches)
    return tables

def extract_equations(tex_content):
    """Extract key equations (LoRA's W = W0 + BA, etc.)."""
    if not tex_content:
        return []
    
    patterns = [
        r'\\begin\{equation\}(.*?)\\end\{equation\}',
        r'\\begin\{equation\*\}(.*?)\\end\{equation\*\}',
        r'\\begin\{align\}(.*?)\\end\{align\}',
        r'\\begin\{align\*\}(.*?)\\end\{align\*\}',
    ]
    equations = []
    for pattern in patterns:
        matches = re.findall(pattern, tex_content, re.DOTALL | re.IGNORECASE)
        equations.extend(matches)
    return equations

def extract_arxiv_references(tex_content):
    """Return the arXiv identifiers a paper cites.

    Bibliographies write identifiers in several shapes — "arXiv:2402.03300",
    a link, or a DOI — and inside the reference list a bare identifier is
    unambiguous. Outside it, a bare four-digit-dot-number would also match
    section and equation numbering, so bare matching is confined to the
    bibliography half of the source.
    """
    body, bib = split_source(tex_content)
    ids = set()
    explicit = [
        r'arxiv[:/\s]*(\d{4}\.\d{4,5})',
        r'arxiv\.org/(?:abs|pdf)/(\d{4}\.\d{4,5})',
        r'10\.48550/arxiv\.(\d{4}\.\d{4,5})',
    ]
    for pattern in explicit:
        ids |= set(re.findall(pattern, body + bib, re.I))
    ids |= set(re.findall(r'(\d{4}\.\d{4,5})', bib))
    return ids


def extract_sections(tex_content):
    """Extract experimental-setup and limitations/conclusion prose sections.

    Abstracts state claims and tables state numbers; the setup sections say
    which model sizes and rewards the numbers came from, and the limitation /
    conclusion sections say where the method breaks. Both matter when the
    debaters weigh evidence, so they are indexed as their own content types.
    """
    if not tex_content:
        return []
    SECTION_KINDS = [
        ('setup', ('experiment', 'setup', 'training detail', 'implementation detail', 'evaluation')),
        ('discussion', ('limitation', 'discussion', 'conclusion', 'future work')),
    ]
    heads = list(re.finditer(r'\\(?:sub)?section\*?\{([^}]*)\}', tex_content))
    sections = []
    for idx, m in enumerate(heads):
        title = m.group(1)
        kind = None
        for k, keys in SECTION_KINDS:
            if any(key in title.lower() for key in keys):
                kind = k
                break
        if not kind:
            continue
        start = m.end()
        end = heads[idx + 1].start() if idx + 1 < len(heads) else min(len(tex_content), start + 20000)
        body = tex_content[start:end]
        # Tables, figures and algorithms are indexed separately; keep the prose.
        body = re.sub(r'\\begin\{(figure|table|tabular|algorithm|algorithmic)\*?\}.*?\\end\{\1\*?\}',
                      ' ', body, flags=re.DOTALL | re.IGNORECASE)
        body = body.strip()
        if len(body) < 300:
            continue
        sections.append((kind, title, body[:12000]))
    return sections

# ═══════════════════════════════════════════════════════════════════════════════
# Anchor papers, fetched by identifier
# ═══════════════════════════════════════════════════════════════════════════════
# Keyword search alone cannot reach these. arXiv ranks relevance by term match,
# weighted towards titles and with no notion of influence, so a search for
# "group relative policy optimization" returns dozens of 2025-2026 papers with
# the phrase in their title while the paper that introduced the method — which
# mentions it only in its abstract — is absent from the first 300 results.
#
# Two kinds of paper are worth naming explicitly:
#   - definitional: the work that introduced a method defines what is being
#     debated, and every derivative paper argues against its formulation;
#   - counter-evidence: a method's own paper never reports the setting where it
#     loses, so the debate needs the work that tested it and disagreed.
# Fetching by id_list is exact, so both arrive regardless of ranking.
ANCHOR_PAPERS = {
    "2402.03300": "DeepSeekMath — introduces GRPO (definitional)",
    "2501.12948": "DeepSeek-R1 — RL-trained reasoning, and its distilled variants (definitional)",
    "2504.13837": "Does RL Really Incentivize Reasoning Capacity? — pass@k evidence that RLVR sharpens rather than creates (counter-evidence)",
}

info("Fetching anchor papers by identifier...")
for i, result in enumerate(client.results(arxiv.Search(id_list=list(ANCHOR_PAPERS)))):
    paper_id = result.entry_id.split('/')[-1]
    papers.append({
        'paper_id': paper_id,
        'title': result.title,
        'authors': ', '.join([a.name for a in result.authors]),
        'published': result.published.strftime('%Y-%m-%d'),
        'summary': result.summary,
        'full_text': f"Title: {result.title}\n\nAbstract: {result.summary}",
        'category': 'anchor'
    })
    info(f"  {result.title[:60]}")

anchor_count = len([p for p in papers if p['category'] == 'anchor'])
info(f"Fetched {anchor_count} anchor papers")

# ═══════════════════════════════════════════════════════════════════════════════
# Fetch GRPO/RLVR papers (Pro-GRPO evidence)
# ═══════════════════════════════════════════════════════════════════════════════
info("Fetching GRPO/RLVR papers (Pro-GRPO evidence)...")
search_grpo = arxiv.Search(
    query=query_grpo,
    max_results=50,
    sort_by=arxiv.SortCriterion.Relevance
)

for i, result in enumerate(client.results(search_grpo)):
    if any(p['paper_id'].split('v')[0] == result.entry_id.split('/')[-1].split('v')[0] for p in papers):
        continue  # already present as an anchor
    papers.append({
        'paper_id': result.entry_id.split('/')[-1],
        'title': result.title,
        'authors': ', '.join([a.name for a in result.authors]),
        'published': result.published.strftime('%Y-%m-%d'),
        'summary': result.summary,
        'full_text': f"Title: {result.title}\n\nAbstract: {result.summary}",
        'category': 'grpo'
    })
    if (i + 1) % 10 == 0:
        info(f"  Fetched {i + 1} GRPO/RLVR papers...")

grpo_count = len([p for p in papers if p['category'] == 'grpo'])
info(f"Fetched {grpo_count} GRPO/RLVR papers")

# ═══════════════════════════════════════════════════════════════════════════════
# Fetch reasoning-distillation papers (Pro-Distillation evidence)
# ═══════════════════════════════════════════════════════════════════════════════
info("Fetching reasoning-distillation papers (Pro-Distillation evidence)...")
search_distill = arxiv.Search(
    query=query_distill,
    max_results=50,
    sort_by=arxiv.SortCriterion.Relevance
)

for i, result in enumerate(client.results(search_distill)):
    paper_id = result.entry_id.split('/')[-1]
    # Avoid duplicates
    if not any(p['paper_id'] == paper_id for p in papers):
        papers.append({
            'paper_id': paper_id,
            'title': result.title,
            'authors': ', '.join([a.name for a in result.authors]),
            'published': result.published.strftime('%Y-%m-%d'),
            'summary': result.summary,
            'full_text': f"Title: {result.title}\n\nAbstract: {result.summary}",
            'category': 'distill'
        })
    if (i + 1) % 10 == 0:
        info(f"  Processed {i + 1} distillation papers...")

distill_count = len([p for p in papers if p['category'] == 'distill'])
info(f"Fetched {distill_count} distillation papers (after deduplication)")

# ═══════════════════════════════════════════════════════════════════════════════
# Fetch general reasoning post-training papers (background knowledge)
# ═══════════════════════════════════════════════════════════════════════════════
info("Fetching general reasoning post-training papers (background knowledge)...")
search_general = arxiv.Search(
    query=query_general,
    max_results=50,
    sort_by=arxiv.SortCriterion.Relevance
)

for i, result in enumerate(client.results(search_general)):
    paper_id = result.entry_id.split('/')[-1]
    if not any(p['paper_id'] == paper_id for p in papers):
        papers.append({
            'paper_id': paper_id,
            'title': result.title,
            'authors': ', '.join([a.name for a in result.authors]),
            'published': result.published.strftime('%Y-%m-%d'),
            'summary': result.summary,
            'full_text': f"Title: {result.title}\n\nAbstract: {result.summary}",
            'category': 'general'
        })
    if (i + 1) % 10 == 0:
        info(f"  Processed {i + 1} general papers...")

general_count = len([p for p in papers if p['category'] == 'general'])
info(f"Fetched {general_count} general reasoning papers (after deduplication)")

# ═══════════════════════════════════════════════════════════════════════════════
# Download LaTeX source for algorithm extraction (with caching)
# ═══════════════════════════════════════════════════════════════════════════════
if FETCH_FULL_TEXT:
    info("Downloading LaTeX source for algorithm extraction...")
    info("(Using local cache - subsequent runs will be much faster)")
else:
    info("FETCH_FULL_TEXT is off — indexing abstracts only")
    info("(set FETCH_FULL_TEXT = True above to also extract algorithms, tables, equations and prose sections)")

papers_with_latex = 0
papers_from_cache = 0
total_algorithms = 0
total_tables = 0
total_equations = 0
total_sections = 0

for i, paper in enumerate(papers):
    # Check if already cached
    cache_path = get_latex_cache_path(paper['paper_id'])
    from_cache = os.path.exists(cache_path)
    
    tex_content = download_latex_source(paper['paper_id']) if FETCH_FULL_TEXT else None
    if tex_content:
        body, _ = split_source(tex_content)
        paper['latex_source'] = tex_content
        # Extractors read the body only: a bibliography is not a prose section.
        paper['algorithms'] = extract_algorithms(body)
        paper['tables'] = extract_tables(body)
        paper['equations'] = extract_equations(body)
        paper['sections'] = extract_sections(body)
        paper['references'] = extract_arxiv_references(tex_content)
        papers_with_latex += 1
        if from_cache:
            papers_from_cache += 1
        total_algorithms += len(paper['algorithms'])
        total_tables += len(paper['tables'])
        total_equations += len(paper['equations'])
        total_sections += len(paper['sections'])
    else:
        paper['latex_source'] = None
        paper['algorithms'] = []
        paper['tables'] = []
        paper['equations'] = []
        paper['sections'] = []
        paper['references'] = set()
    
    if (i + 1) % 20 == 0:
        info(f"  Processed {i + 1}/{len(papers)} papers for LaTeX...")
    
    # Rate limit per ArXiv API Terms of Use (no more than 1 request per 3 seconds)
    # Skip delay if loaded from cache, or if we never made a request
    if FETCH_FULL_TEXT and not from_cache:
        time.sleep(3)

success(f"Total: {len(papers)} papers from ArXiv")
info(f"  - Anchor papers: {anchor_count}")
info(f"  - GRPO/RLVR papers: {grpo_count}")
info(f"  - Distillation papers: {distill_count}")
info(f"  - General reasoning papers: {general_count}")
info(f"  - Papers with LaTeX source: {papers_with_latex} ({papers_from_cache} from cache)")
info(f"  - Total algorithms extracted: {total_algorithms}")
info(f"  - Total tables extracted: {total_tables}")
info(f"  - Total equations extracted: {total_equations}")
info(f"  - Total prose sections extracted (setup/discussion): {total_sections}")
total_refs = sum(len(p.get('references', ())) for p in papers)
info(f"  - Total arXiv references parsed from bibliographies: {total_refs}")

total_chars = sum(len(p['full_text']) for p in papers)
info(f"Total abstract content: {total_chars:,} characters ({total_chars/1000:.1f}K)")

# ArXiv attribution (required by Terms of Use)
print("\n📜 Thank you to arXiv for use of its open access interoperability.")

---
## 2. Let the Corpus Name Its Own Foundations

**What**: Read the bibliographies of the papers just fetched, and add the works
they all build on.

**Why**: keyword search cannot reach a field's foundational papers.

arXiv ranks results by term match, weighted towards titles, with no notion of
influence. Search for *"group relative policy optimization"* and you get dozens
of recent papers with that phrase in their title. The paper that **introduced**
the method mentions it only in its abstract body, and never appears — not in the
first fifty results, not in the first three hundred.

Ranking by citations would fix this, and two public services offer citation
data. Both were tested here, and both were rejected:

| source | licence | why not |
|---|---|---|
| OpenAlex | CC0 — ideal | no reference data for arXiv preprints; the paper that introduced GRPO shows 84 citations and zero references, and DeepSeek-R1 is absent entirely |
| Semantic Scholar | ODC-BY | has the data, but needs an API key tied to an affiliation — a poor thing to require of everyone who runs this |

The answer was already on disk. **Every paper carries its own bibliography**,
inside the archive already downloaded. Reading it costs no request, needs no
key, and adds no licence obligation. And a work cited by most of the corpus is,
by the corpus's own account, foundational to it.

**How**: tally every arXiv identifier cited across all papers, drop those already
indexed, fetch the most-cited remainder by identifier, and label them `anchor`.

**Does it work?** On this corpus the top of that ranking is DeepSeekMath (cited
by 89 of 145 papers), DeepSeek-R1 (88), and PPO (69) — the paper that introduced
GRPO, the model that made it famous, and the algorithm it modifies. The first
two were also on the hand-written anchor list, which is the real evidence: the
method finds what an expert would have chosen, so it also works on a topic where
nobody knows the canon in advance.

**A limit worth stating**: these are *backward* citations — what the corpus was
built on. It reliably surfaces foundations, and only incidentally surfaces later
work that disagrees, since authors do cite the critiques they answer. Papers
that refute a method are found by naming them, as the anchor list does.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Discover anchors from the corpus's own bibliographies
# ═══════════════════════════════════════════════════════════════════════════════
# Requires FETCH_FULL_TEXT — without the sources there are no reference lists.
N_DISCOVERED_ANCHORS = 6

import collections

reference_counts = collections.Counter()
for paper in papers:
    for ref in paper.get('references', ()):
        reference_counts[ref] += 1

already_indexed = {p['paper_id'].split('v')[0] for p in papers}
missing = [(aid, n) for aid, n in reference_counts.most_common()
           if aid not in already_indexed]

if not reference_counts:
    info("No references parsed — set FETCH_FULL_TEXT = True and re-run the fetch cell")
else:
    info(f"{len(reference_counts)} distinct works cited across {len(papers)} papers")
    info(f"{len(missing)} of them are not in the corpus")
    info("")
    info("Most-cited works missing from the corpus:")

    to_fetch = [aid for aid, _ in missing[:N_DISCOVERED_ANCHORS]]
    discovered = list(client.results(arxiv.Search(id_list=to_fetch)))
    citation_count = dict(missing[:N_DISCOVERED_ANCHORS])

    for result in discovered:
        paper_id = result.entry_id.split('/')[-1]
        cites = citation_count.get(paper_id.split('v')[0], 0)
        share = 100 * cites / len(papers)
        info(f"  cited by {cites:>3} of our papers ({share:.0f}%) — {result.title[:58]}")
        papers.append({
            'paper_id': paper_id,
            'title': result.title,
            'authors': ', '.join([a.name for a in result.authors]),
            'published': result.published.strftime('%Y-%m-%d'),
            'summary': result.summary,
            'full_text': f"Title: {result.title}\n\nAbstract: {result.summary}",
            'category': 'anchor',
            'algorithms': [], 'tables': [], 'equations': [], 'sections': [],
            'references': set(), 'latex_source': None,
        })

    # Without their sources these papers enter the index as a single abstract
    # each, and lose every retrieval race against a paper contributing sixty
    # chunks. Under the same opt-in that governs every other source, pull them.
    if FETCH_FULL_TEXT and discovered:
        info("")
        info("Fetching sources for the discovered anchors...")
        for paper in papers[-len(discovered):]:
            cache_path = get_latex_cache_path(paper['paper_id'])
            from_cache = os.path.exists(cache_path)
            tex_content = download_latex_source(paper['paper_id'])
            if tex_content:
                body, _ = split_source(tex_content)
                paper['latex_source'] = tex_content
                paper['algorithms'] = extract_algorithms(body)
                paper['tables'] = extract_tables(body)
                paper['equations'] = extract_equations(body)
                paper['sections'] = extract_sections(body)
                paper['references'] = extract_arxiv_references(tex_content)
            if not from_cache:
                time.sleep(3)  # arXiv Terms of Use
        with_source = sum(1 for p in papers[-len(discovered):] if p.get('latex_source'))
        info(f"  {with_source}/{len(discovered)} anchors have sources indexed")

    success(f"Added {len(discovered)} discovered anchor papers — corpus is now {len(papers)} papers")

---
## 3. Chunk Documents

**What**: Split each paper into overlapping pieces of about 1000 characters,
tagged with the content type they came from.

**Why**: an embedding describes a whole passage with one vector. Feed it a
paper and you get a vague average of everything the paper says; feed it a
paragraph and you get something specific enough to match a question. The 200
character overlap keeps a sentence that straddles a boundary readable in at
least one chunk.

**How**: LangChain's `RecursiveCharacterTextSplitter` tries separators in order —
paragraph breaks, then line breaks, then spaces, then characters — so it cuts at
the most natural boundary available.

### Why the evidence types are capped

A paper contributes a few paragraphs of abstract, but dozens of tables and
equations. Left uncapped, this corpus was **70% numbers**, and questions like
*"does RL create new reasoning ability?"* were answered from benchmark grids
that contained no argument at all.

`MAX_RESULTS_CHUNKS_PER_PAPER` and `MAX_EQUATION_CHUNKS_PER_PAPER` keep the main
results and defining equations of each paper while dropping its appendix sweeps
and inline algebra. The effect on this corpus:

| | uncapped | capped |
|---|---|---|
| results chunks | 3,887 | 1,101 |
| equation chunks | 2,368 | 597 |
| **prose share of the index** | **30%** | **52%** |

Half the index, and better answers — the discarded chunks were mostly hyperparameter
tables that no question ever needed. Raise the caps if your questions are
primarily quantitative.

**Watch the output**: the cell reports the prose share, which is the number to
keep an eye on when you change corpus or caps.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

info("Chunking documents with LangChain...")
info("Creating chunks with content_type metadata for targeted retrieval...")
info("Converting LaTeX to plain text for better embeddings...")

# A paper carries a handful of abstract paragraphs but dozens of tables and
# equations, so an uncapped index is ~70% numbers. Retrieval then answers
# conceptual questions with number soup. Cap the evidence types per paper:
# the main results tables and defining equations survive, the appendix sweeps
# and inline algebra do not.
MAX_RESULTS_CHUNKS_PER_PAPER = 8
MAX_EQUATION_CHUNKS_PER_PAPER = 5

# Create text splitter for chunking documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

for paper in papers:
    paper_results_chunks = 0

    # ═══════════════════════════════════════════════════════════════════════════
    # Abstract chunks (content_type: 'abstract')
    # Abstracts are already in natural language, no conversion needed
    # ═══════════════════════════════════════════════════════════════════════════
    splits = text_splitter.split_text(paper['full_text'])
    
    for i, chunk_text in enumerate(splits):
        chunks.append({
            'paper_id': paper['paper_id'],
            'title': paper['title'],
            'chunk_index': i,
            'text': chunk_text,           # Plain text (for embeddings)
            'text_raw': chunk_text,       # Raw text (same as text for abstracts)
            'published_date': paper['published'],
            'authors': paper['authors'],
            'content_type': 'abstract',
            'category': paper.get('category', 'general')
        })
    
    # ═══════════════════════════════════════════════════════════════════════════
    # Algorithm chunks (content_type: 'algorithm')
    # These contain actual pseudocode/algorithm descriptions from papers
    # Store both raw LaTeX and plain text version for better retrieval
    # ═══════════════════════════════════════════════════════════════════════════
    for j, algo in enumerate(paper.get('algorithms', [])):
        # Raw LaTeX version
        algo_text_raw = f"Algorithm from paper '{paper['title']}':\n\n{algo}"
        # Plain text version for embeddings
        algo_text_plain = f"Algorithm from paper '{paper['title']}':\n\n{latex_to_plain(algo)}"
        
        # Split long algorithms if needed (use plain text for splitting)
        algo_splits_plain = text_splitter.split_text(algo_text_plain)
        algo_splits_raw = text_splitter.split_text(algo_text_raw)
        
        # Pair up plain and raw chunks (may have different counts due to size differences)
        for k, algo_chunk_plain in enumerate(algo_splits_plain):
            # Get corresponding raw chunk if available, otherwise use plain
            algo_chunk_raw = algo_splits_raw[k] if k < len(algo_splits_raw) else algo_chunk_plain
            
            chunks.append({
                'paper_id': paper['paper_id'],
                'title': paper['title'],
                'chunk_index': f"algo-{j}-{k}",
                'text': algo_chunk_plain,     # Plain text (for embeddings)
                'text_raw': algo_chunk_raw,   # Raw LaTeX (for display/code gen)
                'published_date': paper['published'],
                'authors': paper['authors'],
                'content_type': 'algorithm',
                'category': paper.get('category', 'general')
            })
    
    # ═══════════════════════════════════════════════════════════════════════════
    # Results table chunks (content_type: 'results')
    # These contain performance metrics, comparisons, benchmarks
    # CRITICAL: Convert LaTeX to plain text for better embedding retrieval
    # ═══════════════════════════════════════════════════════════════════════════
    for j, table in enumerate(paper.get('tables', [])):
        # Raw LaTeX version
        table_text_raw = f"Results table from paper '{paper['title']}':\n\n{table}"
        # Plain text version for embeddings (converts \textbf{}, &, \\ etc.)
        table_text_plain = f"Results table from paper '{paper['title']}':\n\n{latex_to_plain(table)}"
        
        # Split long tables if needed
        table_splits_plain = text_splitter.split_text(table_text_plain)
        table_splits_raw = text_splitter.split_text(table_text_raw)
        
        for k, table_chunk_plain in enumerate(table_splits_plain):
            if paper_results_chunks >= MAX_RESULTS_CHUNKS_PER_PAPER:
                break
            paper_results_chunks += 1
            table_chunk_raw = table_splits_raw[k] if k < len(table_splits_raw) else table_chunk_plain
            
            chunks.append({
                'paper_id': paper['paper_id'],
                'title': paper['title'],
                'chunk_index': f"table-{j}-{k}",
                'text': table_chunk_plain,    # Plain text (for embeddings)
                'text_raw': table_chunk_raw,  # Raw LaTeX (for display)
                'published_date': paper['published'],
                'authors': paper['authors'],
                'content_type': 'results',
                'category': paper.get('category', 'general')
            })
    
    # ═══════════════════════════════════════════════════════════════════════════
    # Equation chunks (content_type: 'equation')
    # Key mathematical formulations (e.g., the group-relative advantage in GRPO)
    # Keep raw LaTeX for equations since math notation is meaningful
    # ═══════════════════════════════════════════════════════════════════════════
    for j, equation in enumerate(paper.get('equations', [])[:MAX_EQUATION_CHUNKS_PER_PAPER]):
        eq_text_raw = f"Key equation from paper '{paper['title']}':\n\n{equation}"
        eq_text_plain = f"Key equation from paper '{paper['title']}':\n\n{latex_to_plain(equation)}"
        
        chunks.append({
            'paper_id': paper['paper_id'],
            'title': paper['title'],
            'chunk_index': f"eq-{j}",
            'text': eq_text_plain,      # Plain text (for embeddings)
            'text_raw': eq_text_raw,    # Raw LaTeX (for display)
            'published_date': paper['published'],
            'authors': paper['authors'],
            'content_type': 'equation',
            'category': paper.get('category', 'general')
        })

    # ═══════════════════════════════════════════════════════════════════════════
    # Prose section chunks (content_type: 'setup' or 'discussion')
    # Experimental setup says which model sizes / rewards the results used;
    # limitations and conclusions say where the method breaks. Both are
    # evidence the abstracts and tables alone do not carry.
    # ═══════════════════════════════════════════════════════════════════════════
    for j, (sec_kind, sec_title, sec_body) in enumerate(paper.get('sections', [])):
        sec_label = 'Experimental setup' if sec_kind == 'setup' else 'Discussion / limitations'
        sec_text_raw = f"{sec_label} ('{sec_title}') from paper '{paper['title']}':\n\n{sec_body}"
        sec_text_plain = f"{sec_label} ('{sec_title}') from paper '{paper['title']}':\n\n{latex_to_plain(sec_body)}"

        sec_splits_plain = text_splitter.split_text(sec_text_plain)
        sec_splits_raw = text_splitter.split_text(sec_text_raw)

        for k, sec_chunk_plain in enumerate(sec_splits_plain):
            sec_chunk_raw = sec_splits_raw[k] if k < len(sec_splits_raw) else sec_chunk_plain

            chunks.append({
                'paper_id': paper['paper_id'],
                'title': paper['title'],
                'chunk_index': f"sec-{j}-{k}",
                'text': sec_chunk_plain,      # Plain text (for embeddings)
                'text_raw': sec_chunk_raw,    # Raw LaTeX (for display)
                'published_date': paper['published'],
                'authors': paper['authors'],
                'content_type': sec_kind,
                'category': paper.get('category', 'general')
            })

# Count chunks by type
abstract_chunks = len([c for c in chunks if c['content_type'] == 'abstract'])
algorithm_chunks = len([c for c in chunks if c['content_type'] == 'algorithm'])
results_chunks = len([c for c in chunks if c['content_type'] == 'results'])
equation_chunks = len([c for c in chunks if c['content_type'] == 'equation'])
setup_chunks = len([c for c in chunks if c['content_type'] == 'setup'])
discussion_chunks = len([c for c in chunks if c['content_type'] == 'discussion'])

success(f"Created {len(chunks)} total chunks from {len(papers)} papers")
info(f"  - Abstract chunks: {abstract_chunks}")
info(f"  - Algorithm chunks: {algorithm_chunks}")
info(f"  - Results table chunks: {results_chunks}")
info(f"  - Equation chunks: {equation_chunks}")
info(f"  - Setup prose chunks: {setup_chunks}")
info(f"  - Discussion/limitations chunks: {discussion_chunks}")
info(f"Average chunks per paper: {len(chunks) / len(papers):.1f}")
prose = abstract_chunks + setup_chunks + discussion_chunks
info(f"Prose (abstract/setup/discussion) share: {100 * prose / len(chunks):.0f}%")
info("")
info("💡 Each chunk now has 'text' (plain) and 'text_raw' (LaTeX) fields")
info("   - 'text' is used for embedding generation (better retrieval)")
info("   - 'text_raw' is stored in Qdrant for display and code generation")

In [ ]:
import pickle

# Save current papers to cache file
CACHE_FILE = "papers_cache_rl.pkl"
with open(CACHE_FILE, 'wb') as f:
    pickle.dump(papers, f)
print(f"✅ Saved {len(papers)} papers to {CACHE_FILE}")

# Also save chunks if they exist
try:
    with open("chunks_cache_rl.pkl", 'wb') as f:
        pickle.dump(chunks, f)
    print(f"✅ Saved {len(chunks)} chunks to chunks_cache_rl.pkl")
except NameError:
    print("ℹ️ No chunks variable found (not yet created)")

# Save embeddings if they exist
try:
    with open("embeddings_cache_rl.pkl", 'wb') as f:
        pickle.dump(embeddings, f)
    print(f"✅ Saved {len(embeddings)} embeddings to embeddings_cache_rl.pkl")
except NameError:
    print("ℹ️ No embeddings variable found (not yet created)")

---
## 4. Generate Embeddings

**What**: Turn every chunk into a vector, so that "closest in meaning" becomes
"closest in space".

**Why**: keyword search finds the words you typed. Embeddings find the passage
that *means* what you asked, even when it shares no vocabulary with the
question — which is the whole point when a paper says "sharpens the existing
distribution" and you asked "does it create new ability?".

**How**: the same `get_openai_client()` from `tk-llm` serves chat and
embeddings; the model here is whichever embedding model your gateway has
loaded. Each chunk's plain-text form is sent (LaTeX converted), and the
resulting vectors are kept in order alongside the chunks.

### Batching, and the limit that bites

Sending one request per chunk spends nearly all its time on network round-trips.
Sending lists keeps the server busy — several times faster for a corpus of a
few thousand chunks.

`BATCH_SIZE = 32` is not arbitrary. Text-embeddings-inference rejects a batch
above its `max_client_batch_size`, 32 by default, with a bare **422** that
surfaces here as a 503 from the gateway. Larger batches are not faster if they
fail; raise the server's limit first if you want to raise this one.

**One detail that returns in section 6**: the API may return items out of order,
so results are re-sorted by `index` before being appended. A silently shuffled
vector list is a bug you will not see until retrieval starts returning nonsense.

In [ ]:
from langfuse import get_client as get_langfuse_client

# Initialize Langfuse for observability
langfuse_client = get_langfuse_client()

info(f"Generating embeddings for {len(chunks)} chunks...")
info(f"Using model: {EMBED_MODEL}")
info("Using plain text (LaTeX converted) for embedding generation")

# Track the embedding generation as a Langfuse observation
with langfuse_client.start_as_current_observation(
    name="embedding-generation",
    as_type="embedding",
    input={"total_chunks": len(chunks), "model": EMBED_MODEL}
) as obs:
    embeddings = []
    total_tokens = 0

    # One request per BATCH of chunks, not per chunk: the embedding server
    # accepts lists, and round-trips dominate the per-chunk cost. TEI rejects
    # batches above its max_client_batch_size (32 by default) with a 422, so
    # stay at that limit.
    BATCH_SIZE = 32
    for start in range(0, len(chunks), BATCH_SIZE):
        batch = [c['text'] for c in chunks[start:start + BATCH_SIZE]]
        response = oai_client.embeddings.create(
            model=EMBED_MODEL,
            input=batch
        )
        # The API may return items out of order; index restores it.
        for item in sorted(response.data, key=lambda d: d.index):
            embeddings.append(item.embedding)

        if hasattr(response, 'usage') and response.usage:
            total_tokens += response.usage.total_tokens

        done = min(start + BATCH_SIZE, len(chunks))
        if done % 1024 < BATCH_SIZE or done == len(chunks):
            info(f"  Generated {done}/{len(chunks)} embeddings")

    obs.update(output={
        "total_embeddings": len(embeddings),
        "dimension": len(embeddings[0]),
        "total_tokens": total_tokens
    })

langfuse_client.flush()

success(f"All {len(embeddings)} embeddings generated (dimension: {len(embeddings[0])})")
info(f"Total tokens used: {total_tokens:,}")
info(f"Embedding trace sent to Langfuse")

---
## 5. Store in Qdrant

**What**: Persist the vectors, with enough metadata alongside them to filter and
to cite.

**Why**: embeddings are expensive to compute and cheap to store. Qdrant keeps
them across restarts, searches them in milliseconds using HNSW, and — the part
this notebook leans on — lets you *filter* by payload while searching.

**How**:

1. **Create the collection** with the embedding model's dimension and cosine
   distance. Cosine compares direction rather than magnitude, which is what you
   want when a long passage and a short one express the same idea.
2. **Attach a payload** to every vector:

   | field | purpose |
   |---|---|
   | `text` | the plain-text chunk — what the LLM reads |
   | `text_raw` | the original LaTeX — preserved for display and code generation |
   | `content_type` | `abstract`, `algorithm`, `results`, `equation`, `setup`, `discussion` |
   | `category` | `anchor`, `grpo`, `distill`, `general` — which side of the debate, or a foundation |
   | `metadata` | paper id, title, authors, date, chunk index |

3. **Upload in batches of 500.** Qdrant caps a single request at 32 MB, and
   several thousand chunks carrying raw LaTeX exceed that comfortably.

**Why the payload fields earn their keep**: `content_type` is what lets a search
ask for *limitations sections only*; `category` is what lets the debate in
notebook 03 reserve a slot for foundational papers. Both are used, not
decorative — a vector store without payload filtering could not do either.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

# Connect to Qdrant with proper HTTPS configuration
info(f"Connecting to Qdrant at {QDRANT_URL}")

qdrant = QdrantClient(
    url=QDRANT_URL,
    port=443,
    https=True,
    verify=False
)

collection_name = "rl_reasoning_papers"

# Create collection (delete if exists to ensure clean state)
try:
    qdrant.delete_collection(collection_name)
    info(f"Deleted existing collection '{collection_name}'")
except Exception:
    pass  # Collection doesn't exist yet

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=len(embeddings[0]),
        distance=Distance.COSINE
    )
)

success(f"Created collection '{collection_name}'")

# Prepare points for upload
info(f"Preparing {len(chunks)} vectors for upload...")
info("Including both 'text' (plain) and 'text_raw' (LaTeX) in payload...")

points = []
for chunk, embedding in zip(chunks, embeddings):
    point = PointStruct(
        id=str(uuid.uuid4()),
        vector=embedding,
        payload={
            'text': chunk['text'],              # Plain text (for LangChain retrieval)
            'text_raw': chunk.get('text_raw', chunk['text']),  # Raw LaTeX (for code gen)
            'content_type': chunk.get('content_type', 'abstract'),  # For filtering
            'category': chunk.get('category', 'general'),  # grpo, distill, general
            'metadata': {
                'paper_id': chunk['paper_id'],
                'title': chunk['title'],
                'chunk_index': chunk['chunk_index'],
                'published_date': chunk['published_date'],
                'authors': chunk['authors'],
                'content_type': chunk.get('content_type', 'abstract'),
                'category': chunk.get('category', 'general')
            }
        }
    )
    points.append(point)

# ═══════════════════════════════════════════════════════════════════════════════
# Batch upsert to avoid Qdrant's 32MB payload size limit
# With ~10K chunks and LaTeX content, total payload can exceed 100MB
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 500
total_batches = (len(points) + BATCH_SIZE - 1) // BATCH_SIZE

info(f"Uploading {len(points)} vectors in {total_batches} batches (batch size: {BATCH_SIZE})...")

for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    
    qdrant.upsert(
        collection_name=collection_name,
        points=batch
    )
    
    if batch_num % 5 == 0 or batch_num == total_batches:
        info(f"  Uploaded batch {batch_num}/{total_batches} ({min(i + BATCH_SIZE, len(points))}/{len(points)} points)")

final_count = qdrant.get_collection(collection_name).points_count
success(f"Uploaded {final_count} vectors to Qdrant")

# Show breakdown by content_type
info("Content type distribution in Qdrant:")
for ct in ['abstract', 'algorithm', 'results', 'equation', 'setup', 'discussion']:
    count = len([c for c in chunks if c.get('content_type') == ct])
    info(f"  - {ct}: {count} chunks")

info("")
info("💡 Payload structure for each vector:")
info("   - 'text': Plain text (used by LangChain retriever, embeddings)")
info("   - 'text_raw': Raw LaTeX (for display, code generation)")
info("   - 'content_type': abstract|algorithm|results|equation|setup|discussion (for filtering)")
info("   - 'category': grpo|distill|general (debate side)")

---
## 6. Build the RAG Chain

**What**: Wire retrieval and generation together: embed the question, fetch
matching chunks, put them in the prompt, and let the model answer from them.

**Why**: this is the whole idea of RAG — the model is not asked what it knows,
it is asked what these passages say. Answers become checkable, and every claim
carries a source.

**The flow**

```
"Does RLVR create new reasoning ability?"
   ↓
[Embed query]     → a vector, via the same model that embedded the corpus
   ↓
[Search Qdrant]   → 4 prose chunks + 2 evidence chunks
   ↓
[Prompt template] → "Context: <chunks>  Question: <question>"
   ↓
[Generate]        → an answer grounded in those chunks
   ↓
[Return]          → answer + the source documents behind it
```

### Two settings that decide whether this works at all

Both were found by reading the answers rather than the exit status, and both are
the kind of thing that fails quietly.

**`check_embedding_ctx_length=False`** — not cosmetic. By default LangChain
pre-tokenizes the query with **tiktoken**, OpenAI's tokenizer, and sends a list
of integer token ids instead of text. An OpenAI endpoint understands that; any
other embedding server embeds the digits as if they were words. The query vector
then has nothing to do with the question. Measured on this notebook: cosine
similarity of **0.30** between the LangChain query vector and the vector for the
same sentence sent as text — and **1.0** once the flag is set. With it wrong, a
question about GRPO retrieved a paper on detecting data contamination, and the
model correctly reported that the context did not contain an answer.

**`enable_thinking: False`** — reasoning models spend tokens thinking before
they answer, and that thinking is billed against `max_tokens`. A long think
leaves nothing for the visible answer and the chain returns an empty string.
Grounded summarising of retrieved text does not need it; switching it off also
cut each query from about 60 seconds to 12. Models without the switch ignore
the parameter.

### The balanced retriever

Plain `top_k` returns whatever is most similar, and the most numerous content
type wins most slots. `BalancedRetriever` reserves them instead: **4 prose**
chunks to explain, **2 evidence** chunks to substantiate. A question about
mechanism gets prose that argues; a question about performance still gets the
table. It is about thirty lines, and it is the difference between a benchmark
grid and an explanation.

In [17]:
# Fix LangChain version incompatibility if needed
import subprocess
import sys

try:
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    info("LangChain packages already compatible")
except ImportError as e:
    if "ModelProfileRegistry" in str(e):
        info("Fixing LangChain version incompatibility...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "langchain_core>=0.3.30"])
        info("Upgraded langchain_core - please restart the kernel and re-run")
    else:
        raise e

# Install langchain-qdrant if not present (required for qdrant-client v1.18+)
try:
    from langchain_qdrant import QdrantVectorStore
except ImportError:
    info("Installing langchain-qdrant...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langchain-qdrant"])
    from langchain_qdrant import QdrantVectorStore

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate

info("Building RAG chain...")

# Get gateway URL and token from tk-llm client configuration
import os
gateway_url = os.environ.get('LLM_GATEWAY_URL', 'https://llm.' + os.environ.get('DOMAIN_NAME', 'thinkube.com'))
gateway_token = os.environ.get('THINKUBE_API_TOKEN', 'not-needed')

# Create LangChain embeddings model pointing to the LLM Gateway
# check_embedding_ctx_length=False is required, not cosmetic: with the default
# True, LangChain pre-tokenizes the query with tiktoken (OpenAI's tokenizer)
# and sends integer token arrays instead of text. A non-OpenAI embedding server
# embeds those arrays as if they were text, so the query vector is unrelated to
# the question and retrieval returns near-random chunks.
embeddings_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    openai_api_base=gateway_url + "/v1",
    openai_api_key=gateway_token,
    check_embedding_ctx_length=False
)

# Create Qdrant vector store (QdrantVectorStore supports qdrant-client v1.18+)
vectorstore = QdrantVectorStore(
    client=qdrant,
    collection_name=collection_name,
    embedding=embeddings_model,
    content_payload_key="text",
    metadata_payload_key="metadata"
)

# LLM Configuration — same gateway, same token
# Reasoning models (Qwen3 family) think before answering and the thinking is
# billed against max_tokens; a long think leaves nothing for the visible answer
# and the chain returns an empty string. Grounded summarisation of retrieved
# excerpts does not need it, so thinking is switched off — models without the
# switch ignore the kwarg.
llm = ChatOpenAI(
    model=CHAT_MODEL,
    openai_api_base=gateway_url + "/v1",
    openai_api_key=gateway_token,
    temperature=0.7,
    max_tokens=2000,
    request_timeout=120,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)

system_message = SystemMessagePromptTemplate.from_template(
    """You are a helpful AI research assistant. Provide clear, complete answers based on the research paper context provided."""
)

human_message = HumanMessagePromptTemplate.from_template(
    """Based on the following research paper excerpts, answer the question in 2-3 concise paragraphs.

Context:
{context}

Question: {question}

Answer:"""
)

chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message
])

# ═══════════════════════════════════════════════════════════════════════════════
# Balanced retriever
# ═══════════════════════════════════════════════════════════════════════════════
# Plain top-k retrieval answers with whatever content type happens to be most
# similar, and tables and equations outnumber prose even after the ingestion
# cap. A question like "does RL create new ability?" is then answered from a
# benchmark grid. This retriever reserves slots: prose (abstract / setup /
# discussion) explains, evidence (results / algorithm / equation) substantiates,
# and both reach the prompt.
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from qdrant_client.models import Filter, FieldCondition, MatchAny

PROSE_TYPES = ["abstract", "setup", "discussion"]
EVIDENCE_TYPES = ["results", "algorithm", "equation"]


class BalancedRetriever(BaseRetriever):
    """Retrieve a fixed mix of explanatory prose and supporting evidence."""

    client: object
    collection: str
    embeddings: object
    prose_k: int = 4
    evidence_k: int = 2

    def _search(self, vector, types, k) -> List[Document]:
        hits = self.client.query_points(
            collection_name=self.collection,
            query=vector,
            query_filter=Filter(must=[FieldCondition(
                key="content_type", match=MatchAny(any=types))]),
            limit=k,
            with_payload=True,
        ).points
        return [
            Document(
                page_content=h.payload.get("text", ""),
                metadata={**h.payload.get("metadata", {}),
                          "content_type": h.payload.get("content_type"),
                          "score": h.score},
            )
            for h in hits
        ]

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        vector = self.embeddings.embed_query(query)
        return (self._search(vector, PROSE_TYPES, self.prose_k)
                + self._search(vector, EVIDENCE_TYPES, self.evidence_k))


retriever = BalancedRetriever(
    client=qdrant,
    collection=collection_name,
    embeddings=embeddings_model,
    prose_k=4,
    evidence_k=2,
)

# Create RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": chat_prompt},
    return_source_documents=True
)

success("RAG chain created successfully")
info(f"Chat model: {CHAT_MODEL}")
info(f"Embedding model: {EMBED_MODEL}")
info(f"Retriever: {retriever.prose_k} prose chunks + {retriever.evidence_k} evidence chunks per query")

---
## 7. Add Langfuse Tracing

**What**: Record what the pipeline retrieved and generated, for every query.

**Why**: when a RAG answer is wrong, there are two suspects — retrieval brought
the wrong passages, or the model misread the right ones. You cannot tell which
from the answer alone. A trace shows the chunks that were actually in the
prompt, which settles it immediately.

Both defects fixed in section 6 were diagnosed this way: the retrieved sources
were visibly unrelated to the question, which pointed at the query vector rather
than at the model.

**How**: Langfuse's LangChain `CallbackHandler` attaches to the chain and
captures each LLM call, each retrieval, and the timing of both. Traces carry
`user_id` and `session_id` so a run can be found again later.

**View traces at**: `{LANGFUSE_HOST}` → Traces tab.

In [18]:
# Initialize Langfuse for observability
# Using the LangChain CallbackHandler for automatic tracing
try:
    from langfuse import get_client
    from langfuse.langchain import CallbackHandler
    
    # Initialize the Langfuse client
    langfuse_client = get_client()
    
    # Test the connection
    langfuse_client.auth_check()
    
    # Create the LangChain callback handler
    langfuse_handler = CallbackHandler()
    
    success("Langfuse client initialized and connected")
    info(f"Traces will be sent to: {LANGFUSE_HOST}")
    info("Using LangChain CallbackHandler for automatic tracing")
    
except Exception as e:
    error(f"Langfuse initialization failed: {e}")
    langfuse_client = None
    langfuse_handler = None

---
## 8. Test the RAG Pipeline

**What**: Ask three questions and read the answers *and* their sources.

**Why**: "the cell ran" is not "the answer is good". These three are chosen to
fail differently, so the output tells you something:

| question | what it tests |
|---|---|
| *What is GRPO, and how does it differ from PPO?* | definitional retrieval — is the founding work reachable? |
| *Does RLVR create new reasoning ability, or sharpen what exists?* | **contested ground** — does the corpus carry both sides? |
| *When does distillation beat RL on a small model?* | synthesis across several papers with conflicting results |

**How to judge the output** — read the sources, not just the prose:

- **Are the sources on topic?** A fluent answer over unrelated papers means
  retrieval is broken, not the model. That is exactly how the tiktoken defect
  in section 6 was found.
- **Does question 2 present both sides?** It should report a disagreement and
  explain the pass@k evidence: RLVR-trained models win at small *k*, base models
  catch up at large *k*. An answer that confidently picks one side means the
  corpus is missing the other — which was true here until the anchors were
  added.
- **Do the content types vary?** Sources labelled *Experimental setup* and
  *Discussion / limitations* show the balanced retriever doing its job.

**Then open Langfuse** to see retrieval scores, per-step latency and the exact
prompt. And ask your own questions — anything the indexed papers cover.

In [ ]:
import time
import re
from IPython.display import display, Markdown
import os
from langfuse import propagate_attributes

def fix_latex_syntax(text):
    r"""Convert LaTeX from \(...\) and \[...\] to $...$ and $$...$$ for Jupyter"""
    text = re.sub(r'\\\((.*?)\\\)', r'$\1$', text)
    text = re.sub(r'\\\[(.*?)\\\]', r'$$\1$$', text, flags=re.DOTALL)
    return text

# Get user information for Langfuse (construct full email)
jupyter_user = os.environ.get('JUPYTERHUB_USER', 'jupyter-user')
domain_name = os.environ.get('DOMAIN_NAME', 'localhost')
user_id = f"{jupyter_user}@{domain_name}"
session_id = f"rag-session-{int(time.time())}"

# Test queries - aligned with the LoRA papers we indexed
test_queries = [
    "What is GRPO and how does it differ from PPO for training language models?",
    "Does reinforcement learning with verifiable rewards create new reasoning ability, or mostly sharpen what the base model already can do?",
    "When does distilling reasoning traces from a large teacher beat running RL directly on a small model?"
]

# Build markdown output
markdown_output = f"""# 🔬 RAG Pipeline Test Results

**Session Info:**
- User: `{user_id}`
- Session ID: `{session_id}`
- Queries: {len(test_queries)}

---

"""

for i, query in enumerate(test_queries, 1):
    try:
        start_time = time.time()
        
        # Execute RAG query with Langfuse tracing
        if langfuse_handler:
            with propagate_attributes(
                user_id=user_id,
                session_id=session_id,
                tags=["rag-pipeline", "research-assistant", f"query-{i}"]
            ):
                result = rag_chain.invoke(
                    {"query": query},
                    config={
                        "callbacks": [langfuse_handler],
                        "metadata": {
                            "query_number": i,
                            "pipeline": "RAG",
                            "notebook": "02-langchain-rag"
                        }
                    }
                )
        else:
            result = rag_chain.invoke({"query": query})
        
        duration = time.time() - start_time
        
        # Build markdown for this query
        markdown_output += f"""## Query {i}: {query}

⏱️ *Completed in {duration:.2f}s*

### 💡 Answer

{fix_latex_syntax(result['result'])}

### 📚 Sources

"""
        
        # Add sources
        for j, doc in enumerate(result['source_documents'], 1):
            title = doc.metadata.get('title', 'Unknown')[:80]
            paper_id = doc.metadata.get('paper_id', 'N/A')
            authors = doc.metadata.get('authors', 'N/A')
            chunk_preview = doc.page_content[:200].replace('\n', ' ')
            
            markdown_output += f"""**[{j}]** {title}  
- **Paper ID:** `{paper_id}`
- **Authors:** {authors}
- **Excerpt:** *"{chunk_preview}..."*

"""
        
        markdown_output += "\n---\n\n"
    
    except Exception as e:
        markdown_output += f"""## Query {i}: {query}

❌ **Error:** {str(e)}

---

"""

# Flush traces to Langfuse
if langfuse_client:
    langfuse_client.flush()
    markdown_output += f"""## 📊 Observability

✅ All traces sent to Langfuse: [{LANGFUSE_HOST}]({LANGFUSE_HOST})

**Filter by:**
- User ID: `{user_id}`
- Session ID: `{session_id}`
"""

# Display as single markdown block
display(Markdown(markdown_output))

---
## 9. Visualise the Traces

**What**: Pull the traces back from Langfuse and plot their latencies.

**Why**: individual traces explain one query; the distribution explains the
system. It shows which stage dominates the wall clock, and how much a stage
varies between runs — variance usually matters more than the mean, because it is
what users notice.

**How**: fetch recent traces over the Langfuse HTTP API, group them by operation,
and plot each group separately with Plotly.

**The point of separating them**: embedding a corpus and answering a question
have latencies orders of magnitude apart. Pooling them produces a mean that
describes neither — a distribution with two peaks reported as one number. Always
segment latency by operation before drawing conclusions from it.

In [27]:
import time
import pandas as pd
from datetime import datetime, timedelta, timezone

# Langfuse computes a trace's latency from the observations it has received so
# far. Fetch a trace while its spans are still arriving and it reports a
# fraction of its real duration — a 50-second query has shown up as 22 ms.
# Two defences: give ingestion time, then leave out anything still inside that
# window rather than plotting it as fast.
INGESTION_GRACE_SECONDS = 30

info("Fetching traces from Langfuse...")

# Give the traces from this session time to arrive
time.sleep(5)

# 📚 Educational: Langfuse SDK vs HTTP API
# The Langfuse Python SDK (langfuse_client) is designed for SENDING traces, not fetching them.
# To retrieve/query traces, we use the Langfuse HTTP REST API with the v3 endpoint.
# This is the correct approach for building observability dashboards.

try:
    # Fetch traces using Langfuse v3 API
    # Note: langfuse_client.api.trace.list() accesses the internal API client
    traces = langfuse_client.api.trace.list(limit=50)
    
    if not traces.data:
        info("No traces found yet. Run the RAG queries first!")
        df = pd.DataFrame()
    else:
        success(f"Fetched {len(traces.data)} traces from Langfuse")
        
        # 📊 Extract and structure trace data
        cutoff = datetime.now(timezone.utc) - timedelta(seconds=INGESTION_GRACE_SECONDS)
        trace_data = []
        still_ingesting = 0
        for trace in traces.data:
            # Timestamps come back UTC; older SDK builds omit the tzinfo
            ts = trace.timestamp
            if ts is not None and ts.tzinfo is None:
                ts = ts.replace(tzinfo=timezone.utc)
            if ts is not None and ts > cutoff:
                still_ingesting += 1
                continue

            # Langfuse returns latency in SECONDS - convert to milliseconds for better readability
            latency = trace.latency if hasattr(trace, 'latency') and trace.latency else None
            
            trace_data.append({
                'id': trace.id[:8] if trace.id else 'unknown',
                'name': trace.name or 'unnamed',  # Trace type (RetrievalQA, embedding-generation, etc.)
                'timestamp': trace.timestamp,
                'latency_ms': latency * 1000 if latency else None,  # Convert seconds → milliseconds
                'user_id': trace.user_id,
                'session_id': trace.session_id,
            })
        
        # Say what was dropped. A quietly shortened dataset reads as a complete one.
        if still_ingesting:
            info(f"Excluded {still_ingesting} trace(s) newer than "
                 f"{INGESTION_GRACE_SECONDS}s — their latency is not final yet")
        
        df = pd.DataFrame(trace_data)
        
        if df.empty:
            info("Every trace is still within the ingestion window — "
                 "wait a moment and re-run this cell.")
        else:
            # Display trace summary
            print("\n📊 Trace Summary")
            print("-" * 50)
            print(f"Total traces: {len(df)}")
            print(f"Unique trace types: {df['name'].nunique()}")
            print(f"\nTrace types:")
            print(df['name'].value_counts().to_string())
        
        # 💡 Educational: Why we show trace types
        # Each trace type is a different amount of work — a retrieval-and-answer
        # trace spans vector search and generation, an embedding trace is one
        # model call — so the mix decides what any pooled number means.
        
except Exception as e:
    error(f"Failed to fetch traces: {e}")
    df = pd.DataFrame()


📊 Trace Summary
--------------------------------------------------
Total traces: 14
Unique trace types: 2

Trace types:
name
RetrievalQA             12
embedding-generation     2


In [28]:
# 📊 Create interactive visualizations to explore RAG pipeline performance
# Educational Goal: Show how to properly analyze performance metrics by trace type

if not df.empty and df['latency_ms'].notna().any():
    import plotly.graph_objects as go
    import plotly.express as px
    
    # Filter out traces without latency data
    df_with_latency = df[df['latency_ms'].notna()].copy()
    
    # Define a clean, colorblind-friendly palette
    colors = px.colors.qualitative.Set2
    
    # ═══════════════════════════════════════════════════════════════════════
    # VISUALIZATION 1: Box Plot - Latency Distribution by Trace Type
    # ═══════════════════════════════════════════════════════════════════════
    # Why: Shows that different operations have VASTLY different performance profiles
    # Key Learning: Never mix metrics from different operation types!
    
    fig1 = go.Figure()
    
    for i, trace_type in enumerate(df_with_latency['name'].unique()):
        trace_latencies = df_with_latency[df_with_latency['name'] == trace_type]['latency_ms']
        fig1.add_trace(go.Box(
            y=trace_latencies,
            name=trace_type,
            marker_color=colors[i % len(colors)],
            boxmean='sd',  # Show mean and std deviation
            hovertemplate="<b>%{fullData.name}</b><br>Latency: %{y:.0f} ms<extra></extra>"
        ))
    
    fig1.update_layout(
        title={
            'text': "📊 Latency Distribution by Trace Type",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        yaxis_title="Latency (milliseconds)",
        xaxis_title="Trace Type",
        template="plotly_white",
        height=500,
        showlegend=False,
        hovermode='closest'
    )
    
    fig1.show()
    
    # ═══════════════════════════════════════════════════════════════════════
    # VISUALIZATION 2: Interactive Histogram with Dropdown Filter
    # ═══════════════════════════════════════════════════════════════════════
    # Why: Allows exploring each trace type's distribution independently
    # Key Learning: the pooled histogram has one peak per trace type, which is
    # why a single mean over all of them describes none of them
    
    fig2 = go.Figure()
    
    # Add histogram for all traces (problematic - shows mixed distributions!)
    fig2.add_trace(go.Histogram(
        x=df_with_latency['latency_ms'],
        name='All Traces',
        nbinsx=20,
        visible=True,
        marker_color=colors[0],
        hovertemplate="Latency: %{x:.0f} ms<br>Count: %{y}<extra></extra>"
    ))
    
    # Add histogram for each trace type (proper segmentation)
    for i, trace_type in enumerate(df_with_latency['name'].unique()):
        trace_latencies = df_with_latency[df_with_latency['name'] == trace_type]['latency_ms']
        fig2.add_trace(go.Histogram(
            x=trace_latencies,
            name=trace_type,
            nbinsx=15,
            visible=False,
            marker_color=colors[i % len(colors)],
            hovertemplate=f"<b>{trace_type}</b><br>Latency: %{{x:.0f}} ms<br>Count: %{{y}}<extra></extra>"
        ))
    
    # Create dropdown menu for filtering
    buttons = [dict(
        label='All Traces',
        method='update',
        args=[{'visible': [True] + [False] * len(df_with_latency['name'].unique())},
              {'title': '📈 Latency Histogram - All Traces'}]
    )]
    
    for i, trace_type in enumerate(df_with_latency['name'].unique()):
        visible = [False] * (len(df_with_latency['name'].unique()) + 1)
        visible[i + 1] = True
        buttons.append(dict(
            label=trace_type,
            method='update',
            args=[{'visible': visible},
                  {'title': f'📈 Latency Histogram - {trace_type}'}]
        ))
    
    fig2.update_layout(
        title={
            'text': '📈 Latency Histogram - All Traces',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        xaxis_title='Latency (milliseconds)',
        yaxis_title='Count',
        template="plotly_white",
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top",
            bgcolor="white",
            bordercolor="#ccc",
            borderwidth=1
        )],
        height=450
    )
    
    fig2.show()
    
    # ═══════════════════════════════════════════════════════════════════════
    # VISUALIZATION 3: Timeline - Latency Over Time
    # ═══════════════════════════════════════════════════════════════════════
    # Why: Detect performance degradation or improvements over time
    # Key Learning: Watch for trends, spikes, or degradation patterns
    
    fig3 = go.Figure()
    
    for i, trace_type in enumerate(df_with_latency['name'].unique()):
        trace_df = df_with_latency[df_with_latency['name'] == trace_type]
        fig3.add_trace(go.Scatter(
            x=trace_df['timestamp'],
            y=trace_df['latency_ms'],
            mode='markers+lines',
            name=trace_type,
            marker=dict(size=10, color=colors[i % len(colors)]),
            line=dict(width=2, color=colors[i % len(colors)]),
            hovertemplate="<b>%{fullData.name}</b><br>Time: %{x}<br>Latency: %{y:.0f} ms<extra></extra>"
        ))
    
    fig3.update_layout(
        title={
            'text': "⏱️ Latency Over Time",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        xaxis_title='Timestamp',
        yaxis_title='Latency (milliseconds)',
        template="plotly_white",
        height=450,
        hovermode='closest',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig3.show()
    
    # ═══════════════════════════════════════════════════════════════════════
    # SUMMARY STATISTICS TABLE
    # ═══════════════════════════════════════════════════════════════════════
    # Educational: Show statistics PER trace type (not mixed!)
    
    print("\n" + "="*80)
    print("📈 PERFORMANCE SUMMARY")
    print("="*80)
    
    for trace_type in df_with_latency['name'].unique():
        latencies = df_with_latency[df_with_latency['name'] == trace_type]['latency_ms']
        if len(latencies) > 0:
            print(f"\n🔹 {trace_type}")
            print(f"   Traces:  {len(latencies)}")
            print(f"   Min:     {latencies.min():.0f} ms  ({latencies.min()/1000:.2f} sec)")
            print(f"   Max:     {latencies.max():.0f} ms  ({latencies.max()/1000:.2f} sec)")
            print(f"   Mean:    {latencies.mean():.0f} ms  ({latencies.mean()/1000:.2f} sec)")
            print(f"   Median:  {latencies.median():.0f} ms  ({latencies.median()/1000:.2f} sec)")
            print(f"   Std Dev: {latencies.std():.0f} ms")
    
    # The comparison is computed from the traces just fetched. A sentence with
    # the numbers typed into it goes stale the first time the pipeline changes,
    # and then contradicts the table printed directly above it.
    medians = df_with_latency.groupby('name')['latency_ms'].median().sort_values()
    print("\n" + "="*80)
    if len(medians) >= 2 and medians.iloc[0] > 0:
        print(f"\n💡 Measured: {medians.index[-1]} has the higher median latency at "
              f"{medians.iloc[-1]/1000:.1f} s, against {medians.iloc[0]/1000:.1f} s for "
              f"{medians.index[0]} — a factor of {medians.iloc[-1]/medians.iloc[0]:.1f}.")
    elif len(medians) == 1:
        print(f"\n💡 Measured: only {medians.index[0]} was traced, "
              f"median {medians.iloc[0]/1000:.1f} s.")
    print("   Each trace type is a different amount of work, so compare a type")
    print("   against itself over time — never one type against another.")
    print("\n" + "="*80)
    
    success(f"View detailed traces at {LANGFUSE_HOST}")
else:
    info("No trace data available for visualization. Run the RAG queries first!")


📈 PERFORMANCE SUMMARY

🔹 RetrievalQA
   Traces:  12
   Min:     12 ms  (0.01 sec)
   Max:     95217 ms  (95.22 sec)
   Mean:    14136 ms  (14.14 sec)
   Median:  22 ms  (0.02 sec)
   Std Dev: 31958 ms

🔹 embedding-generation
   Traces:  2
   Min:     133107 ms  (133.11 sec)
   Max:     134012 ms  (134.01 sec)
   Mean:    133560 ms  (133.56 sec)
   Median:  133560 ms  (133.56 sec)
   Std Dev: 640 ms


💡 Key Takeaway: RetrievalQA (~11s) is much slower than embedding-generation (~1.7s)
   because it involves vector search + LLM generation, while embeddings are just
   one model call. This is normal and expected!



---
## 10. Papers Indexed

Everything now searchable, with its category — `anchor` for the foundational and
counter-evidence papers, `grpo` / `distill` for the two sides, `general` for
background.

Notebook 03 reads this same collection and hands it to debating agents. If you
change the corpus — different queries, different anchors, different caps — the
debate downstream changes with it, which is the point.

In [22]:
# Display all indexed papers for reference
from IPython.display import display, Markdown

papers_md = "| # | Title | Authors | Date | Paper ID |\n"
papers_md += "|---|-------|---------|------|----------|\n"

for i, paper in enumerate(papers, 1):
    title_short = paper['title'][:60] + "..." if len(paper['title']) > 60 else paper['title']
    authors_short = paper['authors'][:40] + "..." if len(paper['authors']) > 40 else paper['authors']
    papers_md += f"| {i} | {title_short} | {authors_short} | {paper['published']} | `{paper['paper_id']}` |\n"

display(Markdown(papers_md))
print(f"\nTotal: {len(papers)} papers indexed in Qdrant collection '{collection_name}'")

| # | Title | Authors | Date | Paper ID |
|---|-------|---------|------|----------|
| 1 | Post-Optimization Adaptive Rank Allocation for LoRA | Vishnuprasadh Kumaravelu, Sunil Gupta, P... | 2026-04-30 | `2604.27796v1` |
| 2 | IGU-LoRA: Adaptive Rank Allocation via Integrated Gradients ... | Xuan Cui, Huiyue Li, Run Zeng, Yunfei Zh... | 2026-03-14 | `2603.13792v1` |
| 3 | Adaptive Rank Allocation for Federated Parameter-Efficient F... | Fei Wu, Jia Hu, Geyong Min, Shiqiang Wan... | 2025-01-24 | `2501.14406v4` |
| 4 | ARD-LoRA: Dynamic Rank Allocation for Parameter-Efficient Fi... | Haseeb Ullah Khan Shinwari, Muhammad Usa... | 2025-06-23 | `2506.18267v1` |
| 5 | ARA: Adaptive Rank Allocation for Efficient Large Language M... | Lin Xv, Jingsheng Gao, Xian Gao, Ting Li... | 2025-10-22 | `2510.19389v1` |
| 6 | Adaptive Rank Allocation: Speeding Up Modern Transformers wi... | Roberto Garcia, Jerry Liu, Daniel Sorvis... | 2025-03-23 | `2503.18216v2` |
| 7 | HyperAdaLoRA: Accelerating LoRA Rank Allocation During Train... | Hao Zhang, Zhenjia Li, Runfeng Bao, Yifa... | 2025-10-03 | `2510.02630v2` |
| 8 | AdaLoRA-QAT: Adaptive Low-Rank and Quantization-Aware Segmen... | Prantik Deb, Srimanth Dhondy, N. Ramakri... | 2026-04-01 | `2604.01167v1` |
| 9 | AdaLoRA: Adaptive Budget Allocation for Parameter-Efficient ... | Qingru Zhang, Minshuo Chen, Alexander Bu... | 2023-03-18 | `2303.10512v2` |
| 10 | Gradient-Based LoRA Rank Allocation Under GRPO: An Empirical... | Yash Ganpat Sawant | 2026-05-08 | `2605.07366v1` |
| 11 | Dynamic Adaptive Rank Space Exploration for Efficient Sentim... | Hongcheng Ding, Fuzhen Hu, Ruiting Deng,... | 2024-10-22 | `2410.16589v2` |
| 12 | Beyond Low-Rank Tuning: Model Prior-Guided Rank Allocation f... | Chuyan Zhang, Kefan Wang, Yun Gu | 2025-06-30 | `2507.00327v1` |
| 13 | A Sampling-Based Adaptive Rank Approach to the Wigner-Poisso... | Andrew Christlieb, Sining Gong, Jing-Mei... | 2025-06-26 | `2506.21314v1` |
| 14 | Layer-wise dynamic rank for compressing large language model... | Zhendong Mi, Bian Sun, Grace Li Zhang, S... | 2025-09-30 | `2509.25622v2` |
| 15 | DR-LoRA: Dynamic Rank LoRA for Fine-Tuning Mixture-of-Expert... | Guanzhi Deng, Bo Li, Ronghao Chen, Xiuji... | 2026-01-08 | `2601.04823v5` |
| 16 | High-order Adaptive Rank Integrators for Multi-scale Linear ... | William A. Sands, Wei Guo, Jing-Mei Qiu,... | 2024-06-27 | `2406.19479v3` |
| 17 | Factor Fitting, Rank Allocation, and Partitioning in Multile... | Tetiana Parshakova, Trevor Hastie, Eric ... | 2023-10-30 | `2310.19214v2` |
| 18 | RankAdaptor: Hierarchical Rank Allocation for Efficient Fine... | Changhai Zhou, Shijie Han, Lining Yang, ... | 2024-06-22 | `2406.15734v2` |
| 19 | Dynamic Rank, Basis, and Matching | Jan van den Brand, Vishal Kumar, Daniel ... | 2026-05-11 | `2605.09917v1` |
| 20 | MARS: Harmonizing Multimodal Convergence via Adaptive Rank S... | Minkyoung Cho, Insu Jang, Shuowei Jin, Z... | 2026-02-28 | `2603.00720v1` |
| 21 | Efficient Dynamic Rank Aggregation | Morteza Alimi, Hourie Mehrabiun, Alireza... | 2025-09-02 | `2509.02885v1` |
| 22 | Dynamic Rank Adaptation for Vision-Language Models | Jiahui Wang, Qin Xu, Bo Jiang, Bin Luo | 2025-07-08 | `2507.05668v1` |
| 23 | A Semi-Lagrangian Adaptive Rank (SLAR) Method for High-Dimen... | Nanyi Zheng, William A. Sands, Daniel Ha... | 2025-10-28 | `2510.24861v1` |
| 24 | A Semi-Lagrangian Adaptive-Rank (SLAR) Method for Linear Adv... | Nanyi Zheng, Daniel Hayes, Andrew Christ... | 2024-11-27 | `2411.17963v1` |
| 25 | Adaptive Rank, Reduced Forgetting: Continual Learning with D... | Haodong Lu, Chongyang Zhao, Jason Xue, L... | 2024-12-01 | `2412.01004v7` |
| 26 | HyDRA: Hierarchical and Dynamic Rank Adaptation for Mobile V... | Yuanhao Xi, Xiaohuan Bing, Ramin Yahyapo... | 2025-12-20 | `2512.20674v1` |
| 27 | Dynamic Rank Adjustment in Diffusion Policies for Efficient ... | Xiatao Sun, Shuo Yang, Yinxing Chen, Fra... | 2025-02-06 | `2502.03822v3` |
| 28 | Dynamic Rank Adjustment for Accurate and Efficient Neural Ne... | Hyuntak Shin, Aecheon Jung, Sungeun Hong... | 2025-08-12 | `2508.08625v3` |
| 29 | Dynamic Rank Reinforcement Learning for Adaptive Low-Rank Mu... | Caner Erden | 2025-12-17 | `2512.15973v2` |
| 30 | Krylov-based Adaptive-Rank Implicit Time Integrators for Sti... | Hamad El Kahza, William Taitano, Jing-Me... | 2024-04-03 | `2404.03119v1` |
| 31 | Structured Learning of Two-Level Dynamic Rankings | Karthik Raman, Thorsten Joachims, Pannag... | 2011-08-13 | `1108.2754v1` |
| 32 | Sylvester-Preconditioned Adaptive-Rank Implicit Time Integra... | Hamad El Kahza, Jing-Mei Qiu, Luis Chaco... | 2024-10-25 | `2410.19662v3` |
| 33 | An Adaptive-rank Approach with Greedy Sampling for Multi-sca... | William A. Sands, Jing-Mei Qiu, Daniel H... | 2025-05-22 | `2505.17191v2` |
| 34 | A model for efficient dynamical ranking in networks | Andrea Della Vecchia, Kibidi Neocosmos, ... | 2023-07-25 | `2307.13544v2` |
| 35 | Efficient Storage of Fine-Tuned Models via Low-Rank Approxim... | Simo Ryu, Seunghyun Seo, Jaejun Yoo | 2023-05-28 | `2305.18425v1` |
| 36 | Unifying Instance and Panoptic Segmentation with Dynamic Ran... | Hao Chen, Chunhua Shen, Zhi Tian | 2020-11-19 | `2011.09796v1` |
| 37 | AROMA: Autonomous Rank-one Matrix Adaptation | Hao Nan Sheng, Zhi-yong Wang, Mingrui Ya... | 2025-04-06 | `2504.05343v2` |
| 38 | Preference-Based Dynamic Ranking Structure Recognition | Nan Lu, Jian Shi, Xin-Yu Tian | 2025-09-29 | `2509.24493v2` |
| 39 | Don't be so Stief! Learning KV Cache low-rank approximation ... | Luca Benfenati, Matteo Risso, Andrea Van... | 2026-01-29 | `2601.21686v1` |
| 40 | Adaptive Ranking Based Constraint Handling for Explicitly Co... | Naoki Sakamoto, Youhei Akimoto | 2018-11-02 | `1811.00764v3` |
| 41 | Adaptive Ranking-based Sample Selection for Weakly Supervise... | Linxin Song, Jieyu Zhang, Tianxiang Yang... | 2022-10-06 | `2210.03092v2` |
| 42 | TARA Test-by-Adaptive-Ranks for Quantum Anomaly Detection wi... | Davut Emre Tasar, Ceren Ocal Tasar | 2025-12-03 | `2512.04016v1` |
| 43 | ElaLoRA: Elastic & Learnable Low-Rank Adaptation for Efficie... | Huandong Chang, Zicheng Ma, Mingyuan Ma,... | 2025-03-31 | `2504.00254v1` |
| 44 | Optimizing Gross Merchandise Volume via DNN-MAB Dynamic Rank... | Yan Yan, Wentao Guo, Meng Zhao, Jinghe H... | 2017-08-14 | `1708.03993v1` |
| 45 | Sparsely Shared LoRA on Whisper for Child Speech Recognition | Wei Liu, Ying Qin, Zhiyuan Peng, Tan Lee | 2023-09-21 | `2309.11756v2` |
| 46 | Analyzing and Forecasting Success in the Men's Ice Hockey Wo... | Vladimír Holý | 2024-09-09 | `2409.05714v2` |
| 47 | TsqLoRA: Towards Sensitivity and Quality Low-Rank Adaptation... | Yu Chen, Yifei Han, Long Zhang, Yue Du, ... | 2025-09-23 | `2509.18585v1` |
| 48 | HCInfer: An Efficient Inference System via Error Compensatio... | Shen Xu, Xiangwen Zhuge, Zhe Xu, Yingkun... | 2026-05-07 | `2605.05819v1` |
| 49 | FoRA: Low-Rank Adaptation Model beyond Multimodal Siamese Ne... | Weiying Xie, Yusi Zhang, Tianlin Hui, Ji... | 2024-07-23 | `2407.16129v1` |
| 50 | A Bayesian Interpretation of Adaptive Low-Rank Adaptation | Haolin Chen, Philip N. Garner | 2024-09-16 | `2409.10673v2` |
| 51 | PLoRA: Efficient LoRA Hyperparameter Tuning for Large Models | Minghao Yan, Zhuang Wang, Zhen Jia, Shiv... | 2025-08-04 | `2508.02932v1` |
| 52 | ALTO: Adaptive LoRA Tuning and Orchestration for Heterogeneo... | Jingwei Zuo, Xinze Feng, Zien Liu, Kaiji... | 2026-04-07 | `2604.05426v2` |
| 53 | A Note on LoRA | Vlad Fomenko, Han Yu, Jongho Lee, Stanle... | 2024-04-07 | `2404.05086v1` |
| 54 | LangVision-LoRA-NAS: Neural Architecture Search for Variable... | Krishna Teja Chitty-Venkata, Murali Eman... | 2025-08-17 | `2508.12512v1` |
| 55 | Efficient Hyper-Parameter Search for LoRA via Language-aided... | Baek Seong-Eun, Lee Jung-Mok, Kim Sung-B... | 2026-01-19 | `2602.11171v1` |
| 56 | MTL-LoRA: Low-Rank Adaptation for Multi-Task Learning | Yaming Yang, Dilxat Muhtar, Yelong Shen,... | 2024-10-12 | `2410.09437v3` |
| 57 | Beyond LoRA vs. Full Fine-Tuning: Gradient-Guided Optimizer ... | Haozhan Tang, Xiuqi Zhu, Xinyin Zhang, B... | 2026-05-08 | `2605.07111v1` |
| 58 | LoRA meets Riemannion: Muon Optimizer for Parametrization-in... | Vladimir Bogachev, Vladimir Aletov, Alex... | 2025-07-16 | `2507.12142v2` |
| 59 | Revisiting LoRA through the Lens of Parameter Redundancy: Sp... | Jiashun Cheng, Aochuan Chen, Nuo Chen, Z... | 2025-06-20 | `2506.16787v1` |
| 60 | UnHype: CLIP-Guided Hypernetworks for Dynamic LoRA Unlearnin... | Piotr Wójcik, Maksym Petrenko, Wojciech ... | 2026-02-03 | `2602.03410v1` |
| 61 | FedEx-LoRA: Exact Aggregation for Federated and Efficient Fi... | Raghav Singhal, Kaustubh Ponkshe, Pranee... | 2024-10-12 | `2410.09432v4` |
| 62 | LoRA is All You Need for Safety Alignment of Reasoning LLMs | Yihao Xue, Baharan Mirzasoleiman | 2025-07-22 | `2507.17075v4` |
| 63 | LoRATK: LoRA Once, Backdoor Everywhere in the Share-and-Play... | Hongyi Liu, Shaochen Zhong, Xintong Sun,... | 2024-02-29 | `2403.00108v2` |
| 64 | Bernoulli-LoRA: A Theoretical Framework for Randomized Low-R... | Igor Sokolov, Abdurakhmon Sadiev, Yury D... | 2025-08-05 | `2508.03820v1` |
| 65 | LD-MoLE: Learnable Dynamic Routing for Mixture of LoRA Exper... | Yuan Zhuang, Yi Shen, Yuexin Bian, Qing ... | 2025-09-30 | `2509.25684v2` |
| 66 | Adaptive Capacity Allocation for Vision Language Action Fine... | Donghoon Kim, Minji Bae, Unghui Nam, Gye... | 2026-03-08 | `2603.07404v1` |
| 67 | Efficient Split Federated Learning for Large Language Models... | Kai Zhao, Zhaohui Yang, Ye Hu, Mingzhe C... | 2025-04-20 | `2504.14667v2` |
| 68 | Singular Value Decomposition on Kronecker Adaptation for Lar... | Yee Hin Chong, Peng Qu | 2025-06-18 | `2506.15251v1` |
| 69 | GoRA: Gradient-driven Adaptive Low Rank Adaptation | Haonan He, Peng Ye, Yuchen Ren, Yuan Yua... | 2025-02-13 | `2502.12171v3` |
| 70 | Extremal properties for dissections of convex 3-polytopes | Jesús A. De Loera, Francisco Santos, Fum... | 2000-12-18 | `0012169v1` |
| 71 | The Primacy of Magnitude in Low-Rank Adaptation | Zicheng Zhang, Haoran Li, Yifeng Zhang, ... | 2025-07-09 | `2507.06558v2` |
| 72 | A second-order-like optimizer with adaptive gradient scaling... | Jérôme Bolte, Ryan Boustany, Edouard Pau... | 2024-10-08 | `2410.05871v2` |
| 73 | GLAD: Generalizable Tuning for Vision-Language Models | Yuqi Peng, Pengfei Wang, Jianzhuang Liu,... | 2025-07-17 | `2507.13089v1` |
| 74 | MSPLoRA: A Multi-Scale Pyramid Low-Rank Adaptation for Effic... | Jiancheng Zhao, Xingda Yu, Zhen Yang | 2025-03-27 | `2503.21838v1` |
| 75 | BeamLoRA: Beam-Constraint Low-Rank Adaptation | Naibin Gu, Zhenyu Zhang, Xiyu Liu, Peng ... | 2025-02-19 | `2502.13604v2` |
| 76 | FouRA: Fourier Low Rank Adaptation | Shubhankar Borse, Shreya Kadambi, Nilesh... | 2024-06-13 | `2406.08798v1` |
| 77 | ConsNoTrainLoRA: Data-driven Weight Initialization of Low-ra... | Debasmit Das, Hyoungwoo Park, Munawar Ha... | 2025-07-09 | `2507.08044v1` |
| 78 | Fine-tuning LLMs with variational Bayesian last layer for hi... | Haotian Xiang, Jinwen Xu, Qin Lu | 2025-10-01 | `2510.01471v2` |
| 79 | Ehrhart polynomials of matroid polytopes and polymatroids | Jesús A. De Loera, David C. Haws, Matthi... | 2007-10-23 | `0710.4346v1` |
| 80 | SEAL: Entangled White-box Watermarks on Low-Rank Adaptation | Giyeong Oh, Saejin Kim, Woohyun Cho, San... | 2025-01-16 | `2501.09284v2` |
| 81 | TASO: Task-Aligned Sparse Optimization for Parameter-Efficie... | Daiye Miao, Yufang Liu, Jie Wang, Changz... | 2025-09-22 | `2509.17688v2` |
| 82 | Parameter Efficient Continual Learning with Dynamic Low-Rank... | Prashant Shivaram Bhat, Shakib Yazdani, ... | 2025-05-17 | `2505.11998v4` |
| 83 | ETHER: Efficient Finetuning of Large-Scale Models with Hyper... | Massimo Bini, Karsten Roth, Zeynep Akata... | 2024-05-30 | `2405.20271v2` |
| 84 | MatryoshkaLoRA: Learning Accurate Hierarchical Low-Rank Repr... | Ionut-Vlad Modoranu, Mher Safaryan, Dan ... | 2026-05-08 | `2605.07850v1` |
| 85 | Origin of third harmonic generation in plasmonic nanoantenna... | Antonino Calà Lesina, Pierre Berini, Lor... | 2016-10-25 | `1610.08025v1` |
| 86 | FedP$^2$EFT: Federated Learning to Personalize PEFT for Mult... | Royson Lee, Minyoung Kim, Fady Rezk, Rui... | 2025-02-05 | `2502.04387v3` |
| 87 | StereoAdapter: Adapting Stereo Depth Estimation to Underwate... | Zhengri Wu, Yiran Wang, Yu Wen, Zeyu Zha... | 2025-09-19 | `2509.16415v1` |
| 88 | GraLoRA: Granular Low-Rank Adaptation for Parameter-Efficien... | Yeonjoon Jung, Daehyun Ahn, Hyungjun Kim... | 2025-05-26 | `2505.20355v2` |
| 89 | Optuna vs Code Llama: Are LLMs a New Paradigm for Hyperparam... | Roman Kochnev, Arash Torabi Goodarzi, Zo... | 2025-04-08 | `2504.06006v4` |
| 90 | FLoRIST: Singular Value Thresholding for Efficient and Accur... | Hariharan Ramesh, Jyotikrishna Dass | 2025-06-10 | `2506.09199v1` |
| 91 | Fed-DLoRA: Efficient Wireless Federated Learning with Dynami... | Huaicheng Li, Junhui Zhao, Haoyu Quan, X... | 2026-04-27 | `2604.24103v1` |
| 92 | SARA: Singular-Value Based Adaptive Low-Rank Adaption | Jihao Gu, Shuai Chen, Zelin Wang, Yibo Z... | 2024-08-06 | `2408.03290v1` |
| 93 | Block Circulant Adapter for Large Language Models | Xinyu Ding, Meiqi Wang, Siyu Liao, Zhong... | 2025-05-01 | `2505.00582v2` |
| 94 | Alternative Chirp Spread Spectrum Techniques for LPWANs | Ivo Bizon Franco de Almeida, Marwa Chafi... | 2021-02-18 | `2102.09250v2` |
| 95 | AdaPaD: Adaptive Parallel Deflation for PEFT with Self-Corre... | Barbara Su, Fangshuo Liao, Anastasios Ky... | 2026-05-11 | `2605.10741v1` |
| 96 | Monotone Paths on Cross-Polytopes | Alexander Black, Jesús De Loera | 2021-02-02 | `2102.01237v2` |
| 97 | Null-LoRA: Low-Rank Adaptation on Null Space | Yi Zhang, Yulei Kang, Haoxuan Chen, Jinx... | 2025-12-17 | `2512.15233v2` |
| 98 | TriAdaptLoRA: Brain-Inspired Triangular Adaptive Low-Rank Ad... | Yao Liang, Yuwei Wang, Yi Zeng | 2025-01-14 | `2501.08008v1` |
| 99 | LoLDU: Low-Rank Adaptation via Lower-Diag-Upper Decompositio... | Yiming Shi, Jiwei Wei, Yujia Wu, Ran Ran... | 2024-10-17 | `2410.13618v1` |
| 100 | MELoRA: Mini-Ensemble Low-Rank Adapters for Parameter-Effici... | Pengjie Ren, Chengshun Shi, Shiguang Wu,... | 2024-02-27 | `2402.17263v3` |
| 101 | VB-LoRA: Extreme Parameter Efficient Fine-Tuning with Vector... | Yang Li, Shaobo Han, Shihao Ji | 2024-05-24 | `2405.15179v3` |
| 102 | LoRA-FAIR: Federated LoRA Fine-Tuning with Aggregation and I... | Jieming Bian, Lei Wang, Letian Zhang, Ji... | 2024-11-22 | `2411.14961v3` |
| 103 | One-for-All: Generalized LoRA for Parameter-Efficient Fine-t... | Arnav Chavan, Zhuang Liu, Deepak Gupta, ... | 2023-06-13 | `2306.07967v2` |
| 104 | OPLoRA: Orthogonal Projection LoRA Prevents Catastrophic For... | Yifeng Xiong, Xiaohui Xie | 2025-10-14 | `2510.13003v2` |
| 105 | Flat-LoRA: Low-Rank Adaptation over a Flat Loss Landscape | Tao Li, Zhengbao He, Yujun Li, Yasheng W... | 2024-09-22 | `2409.14396v2` |
| 106 | Parameter-Efficient Fine-Tuning for HAR: Integrating LoRA an... | Irina Seregina, Philippe Lalanda, German... | 2025-12-19 | `2512.17983v1` |
| 107 | Echo-LoRA: Parameter-Efficient Fine-Tuning via Cross-Layer R... | Yihang Peng, Peng Jin, Jie Gong, Xingyua... | 2026-05-05 | `2605.08177v1` |
| 108 | PC-LoRA: Low-Rank Adaptation for Progressive Model Compressi... | Injoon Hwang, Haewon Park, Youngwan Lee,... | 2024-06-13 | `2406.09117v1` |
| 109 | TT-LoRA MoE: Unifying Parameter-Efficient Fine-Tuning and Sp... | Pradip Kunwar, Minh N. Vu, Maanak Gupta,... | 2025-04-29 | `2504.21190v1` |
| 110 | Why LoRA Fails to Forget: Regularized Low-Rank Adaptation Ag... | Hoang-Chau Luong, Lingwei Chen | 2026-01-09 | `2601.06305v1` |
| 111 | Why LoRA Resists Label Noise: A Theoretical Framework for No... | Brady Steele | 2026-01-22 | `2602.00084v1` |
| 112 | MLAE: Masked LoRA Experts for Visual Parameter-Efficient Fin... | Junjie Wang, Guangjing Yang, Wentao Chen... | 2024-05-29 | `2405.18897v2` |
| 113 | Randomized Asymmetric Chain of LoRA: The First Meaningful Th... | Grigory Malinovsky, Umberto Michieli, Ha... | 2024-10-10 | `2410.08305v1` |
| 114 | LoRA-C: Parameter-Efficient Fine-Tuning of Robust CNN for Io... | Chuntao Ding, Xu Cao, Jianhang Xie, Linl... | 2024-10-22 | `2410.16954v2` |
| 115 | LoRA-Pro: Are Low-Rank Adapters Properly Optimized? | Zhengbo Wang, Jian Liang, Ran He, Zilei ... | 2024-07-25 | `2407.18242v3` |
| 116 | Layer-wise LoRA fine-tuning: a similarity metric approach | Keith Ando Ogawa, Bruno Lopes Yamamoto, ... | 2026-02-05 | `2602.05988v1` |
| 117 | C-LoRA: Contextual Low-Rank Adaptation for Uncertainty Estim... | Amir Hossein Rahmati, Sanket Jantre, Wei... | 2025-05-23 | `2505.17773v3` |
| 118 | Parameter-Efficient Fine-Tuning for Medical Text Summarizati... | Ulugbek Shernazarov, Rostislav Svitsov, ... | 2026-03-23 | `2603.21970v1` |
| 119 | C-LoRA: Continual Low-Rank Adaptation for Pre-trained Models | Xin Zhang, Liang Bai, Xian Yang, Jiye Li... | 2025-02-25 | `2502.17920v1` |
| 120 | NAS-LoRA: Empowering Parameter-Efficient Fine-Tuning for Vis... | Renqi Chen, Haoyang Su, Shixiang Tang | 2025-12-03 | `2512.03499v1` |
| 121 | LoRA-GA: Low-Rank Adaptation with Gradient Approximation | Shaowen Wang, Linxi Yu, Jian Li | 2024-07-06 | `2407.05000v2` |
| 122 | AFLoRA: Adaptive Freezing of Low Rank Adaptation in Paramete... | Zeyu Liu, Souvik Kundu, Anni Li, Junrui ... | 2024-03-20 | `2403.13269v3` |
| 123 | QR-LoRA: QR-Based Low-Rank Adaptation for Efficient Fine-Tun... | Jessica Liang, Anirudh Bharadwaj | 2025-08-29 | `2508.21810v1` |
| 124 | Quantum-PEFT: Ultra parameter-efficient fine-tuning | Toshiaki Koike-Akino, Francesco Tonin, Y... | 2025-03-07 | `2503.05431v1` |
| 125 | BA-LoRA: Bias-Alleviating Low-Rank Adaptation to Mitigate Ca... | Yupeng Chang, Yi Chang, Yuan Wu | 2024-08-08 | `2408.04556v7` |
| 126 | DropLoRA: Sparse Low-Rank Adaptation for Parameter-Efficient... | Haojie Zhang | 2025-08-24 | `2508.17337v1` |
| 127 | DoRA: Enhancing Parameter-Efficient Fine-Tuning with Dynamic... | Yulong Mao, Kaiyu Huang, Changhao Guan, ... | 2024-05-27 | `2405.17357v3` |
| 128 | LoRA-MGPO: Mitigating Double Descent in Low-Rank Adaptation ... | Yupeng Chang, Chenlu Guo, Yi Chang, Yuan... | 2025-02-20 | `2502.14538v3` |
| 129 | BoRA: Bi-dimensional Weight-Decomposed Low-Rank Adaptation | Qiushi Wang, Yuchen Fan, Junwei Bao, Hon... | 2024-12-09 | `2412.06441v1` |
| 130 | Parameter-Efficient Fine-Tuning Design Spaces | Jiaao Chen, Aston Zhang, Xingjian Shi, M... | 2023-01-04 | `2301.01821v1` |
| 131 | RaSA: Rank-Sharing Low-Rank Adaptation | Zhiwei He, Zhaopeng Tu, Xing Wang, Xingy... | 2025-03-16 | `2503.12576v1` |
| 132 | PLoP: Precise LoRA Placement for Efficient Finetuning of Lar... | Soufiane Hayou, Nikhil Ghosh, Bin Yu | 2025-06-25 | `2506.20629v1` |
| 133 | The Expressive Power of Low-Rank Adaptation | Yuchen Zeng, Kangwook Lee | 2023-10-26 | `2310.17513v3` |
| 134 | Put the Space of LoRA Initialization to the Extreme to Prese... | Pengwei Tang, Xiaolin Hu, Yong Liu, Lizh... | 2025-03-04 | `2503.02659v2` |
| 135 | Stable-LoRA: Stabilizing Feature Learning of Low-Rank Adapta... | Yize Wu, Ke Gao, Ling Li, Yanjun Wu | 2026-03-05 | `2603.05204v1` |
| 136 | LoRA: Low-Rank Adaptation of Large Language Models | Edward J. Hu, Yelong Shen, Phillip Walli... | 2021-06-17 | `2106.09685v2` |
| 137 | Hypernetwork-Driven Low-Rank Adaptation Across Attention Hea... | Nghiem T. Diep, Dung Le, Tuan Truong, Ta... | 2025-10-05 | `2510.04295v2` |
| 138 | Less is More: Resource-Efficient Low-Rank Adaptation | Chunlin Tian, Xuyang Wei, Huanrong Liu, ... | 2025-11-30 | `2512.00878v1` |
| 139 | PiCa: Parameter-Efficient Fine-Tuning with Column Space Proj... | Junseo Hwang, Wonguk Cho, Taesup Kim | 2025-05-26 | `2505.20211v3` |
| 140 | MoRA: High-Rank Updating for Parameter-Efficient Fine-Tuning | Ting Jiang, Shaohan Huang, Shengyue Luo,... | 2024-05-20 | `2405.12130v1` |
| 141 | Implicit Style-Content Separation using B-LoRA | Yarden Frenkel, Yael Vinker, Ariel Shami... | 2024-03-21 | `2403.14572v2` |
| 142 | LoRA Done RITE: Robust Invariant Transformation Equilibratio... | Jui-Nan Yen, Si Si, Zhao Meng, Felix Yu,... | 2024-10-27 | `2410.20625v2` |
| 143 | Bayesian-LoRA: LoRA based Parameter Efficient Fine-Tuning us... | Cristian Meo, Ksenia Sycheva, Anirudh Go... | 2024-06-18 | `2406.13046v3` |



Total: 143 papers indexed in Qdrant collection 'research_papers'


In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "show", "langchain-qdrant"], capture_output=True, text=True)
print(result.stdout or "Not installed")
result2 = subprocess.run([sys.executable, "-m", "pip", "show", "langchain-community"], capture_output=True, text=True)
print(result2.stdout[:200])